# CALE L2 Norm Bimodality Survey (g_stock)

**Primary author:** Victoria

**Builds on:**
- *cale_fclue_norm_bimodality.ipynb* (Victoria — establishes that g_stock f_clue norms are visibly bimodal; this notebook generalizes that question across all g_stock embedding populations and adds two word-subset controls)
- *specs/norm_bimodality_survey.md* (Victoria — 7-population scope, ΔBIC + Ashman's D criteria, figure layouts and visual style)
- *specs/norm_bimodality_revision.md* (Victoria — revises §2 visual style to KDE; replaces §3 with ICC and cross-format analyses; adds §4 surface-feature regression)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

This notebook surveys L2 norm distributions across **seven** g_stock
embedding populations — the three phrase types (f_clue, f_wndef, f_wnex)
crossed with full-vs-validation, plus two word-subset controls that hold
the wnex vocabulary fixed and vary the phrase construction — to identify
which exhibit genuine bimodality under a formal two-criterion test
(ΔBIC > 10 AND Ashman's D > 2). Two follow-up sections then probe the
*source* of the f_clue bimodality. §3 asks whether L2 norm is a property
of the word being embedded, using ICC(1,1) on per-word norm groups and a
cross-format Spearman correlation with f_wndef. §4 asks whether
surface-level properties of the f_clue input phrase (subword token
counts, target position, capitalization, punctuation) predict the L2
norm via OLS multiple regression.

The two word-subset populations isolate whether the f_clue bimodality
is driven by the **word subset** (the small wnex vocabulary) or by the
**phrase construction** (clue context vs decontextualized phrases). They
are constructed by *slicing existing norm vectors* — no additional
embedding arrays are loaded.

Reads embedding arrays under `data/embeddings/g_stock/`,
vocabulary/index files under `data/embeddings/g_stock/` and
`data/filtered_split/wn_synset/`, and `f_clue.csv` (for §4 surface-
feature engineering). Writes numerical results to
`outputs/cale_norm_bimodality-results.md` and figures to
`outputs/figures/norm_bimodality_*.png`. Produces no new data
artifacts.


## §0 — Setup

Imports and environment auto-detection. This notebook lives under
`custom_embedding_model/planning/exploration/`, so the component root is
two directories up and the project root is three directories up.
`RANDOM_STATE` is pinned once; all GMM fits and bin edges are
deterministic. The g_stock f_clue full embedding is ~980 MB on disk, so we
load each embedding array briefly to compute its L2 norms and then delete
it — only norm vectors persist past §1.

In [ ]:
# === Imports and configuration
import math
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import norm as sp_norm
from sklearn.mixture import GaussianMixture
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    # planning/exploration/ -> custom_embedding_model/ -> ccc-project/
    PROJECT_ROOT = Path("../../..").resolve()

COMPONENT_ROOT = PROJECT_ROOT / "custom_embedding_model"
DATA_DIR       = COMPONENT_ROOT / "data"
WN_DIR         = DATA_DIR / "filtered_split" / "wn_synset"
EMBED_DIR      = DATA_DIR / "embeddings"
OUTPUT_DIR     = COMPONENT_ROOT / "outputs"
FIG_DIR        = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:   {env_label}")
print(f"PROJECT_ROOT:  {PROJECT_ROOT}")
print(f"EMBED_DIR:     {EMBED_DIR}")
print(f"WN_DIR:        {WN_DIR}")
print(f"OUTPUT_DIR:    {OUTPUT_DIR}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

NOTEBOOK_T0 = time.time()
MODEL_COLORS = {"g_stock": "#1f77b4", "g1": "#ff7f0e"}

In [ ]:
# === Version reporting (Decision 18)
import sklearn

VERSIONS = {
    "python":     sys.version.split()[0],
    "numpy":      np.__version__,
    "pandas":     pd.__version__,
    "scipy":      scipy.__version__,
    "sklearn":    sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seaborn":    sns.__version__,
}
for k, v in VERSIONS.items():
    print(f"{k:12s} {v}")

### §0a — ΔBIC and Ashman's D helper

The Bayesian Information Criterion penalizes model complexity, so comparing
the BIC of a 1-component Gaussian fit against a 2-component mixture gives a
formal criterion for bimodality: ΔBIC = BIC(1) − BIC(2) is positive when
the 2-component model fits better than its extra parameters cost. By
Kass & Raftery (1995), ΔBIC > 10 is "very strong" evidence for the more
complex model.

ΔBIC alone is not enough at large N: it scales with sample size, so any
minor deviation from perfect normality flags as "bimodal" once N is in the
tens of thousands. We pair it with **Ashman's D** (Ashman, Bird & Zepf
1994), a sample-size-independent effect-size measure of how cleanly the
two GMM components are separated:

    D = √2 · |μ₂ − μ₁| / √(σ₁² + σ₂²)

D > 2 is the standard threshold for "cleanly separated" components. A
population is classified as bimodal only when **both** ΔBIC > 10 (the
two-component model is statistically preferred) **and** D > 2 (the modes
are meaningfully separated). The helper returns both metrics plus the
2-component fit parameters (sorted so component 1 has the smaller mean)
for downstream plotting and reporting.

In [ ]:
# === ΔBIC + Ashman's D helper: 1-component vs 2-component GMM
def compute_delta_bic(data, random_state=RANDOM_STATE):
    """Fit 1- and 2-component GMMs; return ΔBIC and Ashman's D.

    ΔBIC = BIC(1) - BIC(2) is positive when 2 components are statistically
    preferred. Ashman's D is the sample-size-independent effect-size measure
    of component separation:

        D = √2 · |μ₂ − μ₁| / √(σ₁² + σ₂²)

    Returns the 2-component GMM fit parameters as well, sorted so component 1
    is always the lower-mean mode.
    """
    X = data.reshape(-1, 1)
    gm1 = GaussianMixture(n_components=1, random_state=random_state).fit(X)
    gm2 = GaussianMixture(n_components=2, random_state=random_state).fit(X)

    bic1 = gm1.bic(X)
    bic2 = gm2.bic(X)
    delta_bic = bic1 - bic2  # positive = 2-component preferred

    # Extract 2-component parameters, sorted by mean so "component 1" is
    # always the lower-mean mode.
    mus = gm2.means_.ravel()
    sigmas = np.sqrt(gm2.covariances_.ravel())
    weights = gm2.weights_
    order = np.argsort(mus)

    # Ashman's D — sample-size-independent measure of how well-separated the
    # two component means are relative to their combined spread. D > 2 is the
    # standard "cleanly separated" threshold (Ashman, Bird & Zepf 1994).
    ashmans_d = (math.sqrt(2) * abs(mus[order[1]] - mus[order[0]])
                 / math.sqrt(sigmas[order[0]]**2 + sigmas[order[1]]**2))

    return {
        "bic_1": bic1,
        "bic_2": bic2,
        "delta_bic": delta_bic,
        "ashmans_d": ashmans_d,
        "mu1": mus[order[0]], "sigma1": sigmas[order[0]],
        "weight1": weights[order[0]],
        "mu2": mus[order[1]], "sigma2": sigmas[order[1]],
        "weight2": weights[order[1]],
    }

## §1 — Load embeddings, compute norms, construct subsets

Five embedding files are loaded directly: f_clue (full and val), f_wndef
(full and val), and f_wnex (full only). The val slice of f_wnex is
omitted because at N≈3,008 it is a small subset of the full wnex
population and would not reveal anything the full version doesn't. Two
additional populations — the wnex-aligned subsets of f_clue and f_wndef
— are constructed in §1b by slicing the already-computed norm vectors,
so the full f_clue array (~980 MB) only needs to be loaded once.

### §1a — Load and compute norms (5 embedding files)

For each array, we load it, assert its expected shape, compute row-wise
L2 norms, and immediately delete the 1024-dim array to keep memory under
control. After this cell only norm vectors (one float per row) remain in
memory. Labels omit the redundant `g_stock · ` prefix — every population
in this survey is g_stock.

In [ ]:
# === Load g_stock embeddings, compute L2 norms, free arrays
norms = {}  # label -> 1D numpy array of L2 norms

# Each tuple: (label, file path, expected shape).
populations = [
    ("f_clue (full)",  EMBED_DIR / "g_stock" / "f_clue.npy",            (239406, 1024)),
    ("f_clue (val)",   EMBED_DIR / "g_stock" / "f_clue_val.npy",        ( 47933, 1024)),
    ("f_wndef (full)", EMBED_DIR / "g_stock" / "f_common_wndef.npy",    ( 53930, 1024)),
    ("f_wndef (val)",  EMBED_DIR / "g_stock" / "f_common_wndef_val.npy",( 26152, 1024)),
    ("f_wnex (full)",  EMBED_DIR / "g_stock" / "f_common_wnex.npy",     (  8360, 1024)),
]

for label, path, expected_shape in populations:
    t0 = time.time()
    arr = np.load(path)
    # Validate shape against DATA.md before computing anything (per project
    # convention — embedding files are committed artifacts so a shape
    # mismatch indicates a stale or corrupted file).
    assert arr.shape == expected_shape, (
        f"{label}: expected {expected_shape}, got {arr.shape}"
    )
    norms[label] = np.linalg.norm(arr, axis=1)
    del arr  # free ~980 MB for f_clue full; keep only the float vector
    print(f"{label:20s} N={len(norms[label]):>7,}  "
          f"loaded+normed in {time.time() - t0:.1f}s")

### §1b — Construct subset populations by slicing

Two control populations isolate **word subset** from **phrase
construction**:

- **f_clue, def ∈ wnex** — the f_clue rows whose definition word is in the
  wnex vocabulary. If f_clue's bimodality persists here, it is not driven
  by definitions outside the wnex vocabulary.
- **f_wndef, wnex words** — the f_wndef rows for the 8,360 wnex-eligible
  words. If this subset is unimodal even though f_wnex (full) is bimodal,
  bimodality is a property of the *phrase construction* (wnex usage
  examples) rather than the *word subset*.

Both subsets are built by slicing the already-computed full norm vectors
— no embedding arrays are reloaded. The row indices come from joining
the f_clue index against `clues_wn_filtered.csv` (for the f_clue subset)
or looking up wnex words in the wndef vocabulary (for the f_wndef
subset).

In [ ]:
# === Subset 1: f_clue rows whose definition is in the wnex vocabulary
# Index/vocab files use keep_default_na=False because crossword words like
# "nan" (grandmother) and numeric strings must not become pandas NaN.
f_clue_index = pd.read_csv(
    EMBED_DIR / "g_stock" / "f_clue_index.csv",
    keep_default_na=False, na_values=[""],
)
clues_wn = pd.read_csv(
    WN_DIR / "clues_wn_filtered.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wnex = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex.csv",
    keep_default_na=False, na_values=[""],
)
wnex_words = set(vocab_wnex["word"])

# Join f_clue_index against clues_wn_filtered to get the WordNet-normalized
# definition (`definition_wn`) for each f_clue row. The merge is on the
# composite key (clue_id, definition); f_clue.csv is built from
# clues_wn_filtered, so this should be a 1:1 join.
merged = f_clue_index.merge(
    clues_wn[["clue_id", "definition", "definition_wn"]],
    on=["clue_id", "definition"],
)
assert len(merged) == len(f_clue_index), (
    f"Expected 1:1 join; got {len(merged)} rows from {len(f_clue_index)}"
)

# Filter to rows where definition_wn is in the wnex vocabulary, then
# extract the row indices that point into the full f_clue.npy / norms vector.
wnex_def_mask = merged["definition_wn"].isin(wnex_words)
wnex_def_rows = merged.loc[wnex_def_mask, "row"].values

norms["f_clue, def ∈ wnex"] = norms["f_clue (full)"][wnex_def_rows]
print(f"f_clue, def ∈ wnex      N={len(norms['f_clue, def ∈ wnex']):>7,}  "
      f"(expected ~79,801)")

In [ ]:
# === Subset 2: f_wndef rows for the 8,360 wnex-eligible words
vocab_wndef = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef.csv",
    keep_default_na=False, na_values=[""],
)
# Word -> row in vocabulary_wndef (which is the index into f_common_wndef.npy).
wndef_word_to_row = dict(zip(vocab_wndef["word"], vocab_wndef["row"]))

# Walk the wnex vocabulary in its canonical order and pull the corresponding
# wndef row index for each word. wnex ⊆ wndef should hold by construction
# (a word with a usage example always also has a definition), so we expect
# all 8,360 wnex words to be present in wndef.
wnex_in_wndef_rows = np.array([
    wndef_word_to_row[w] for w in vocab_wnex["word"] if w in wndef_word_to_row
])
n_missing = len(vocab_wnex) - len(wnex_in_wndef_rows)
if n_missing:
    print(f"WARNING: {n_missing} wnex words not found in wndef vocabulary")

norms["f_wndef, wnex words"] = norms["f_wndef (full)"][wnex_in_wndef_rows]
print(f"f_wndef, wnex words     N={len(norms['f_wndef, wnex words']):>7,}  "
      f"(expected ~8,360)")
print(f"\nAll 7 populations now in norms dict:")
for label in norms:
    print(f"  {label:25s} N={len(norms[label]):>7,}")

### §1c — Per-population norm summary

A side-by-side of N, mean, std, min, and max for all 7 populations. These
are the inputs to every figure and table that follows.

In [ ]:
# === Per-population norm summary table
norm_summary_rows = []
for label, vec in norms.items():
    norm_summary_rows.append({
        "Population": label,
        "N": len(vec),
        "Mean norm": float(vec.mean()),
        "Std norm": float(vec.std()),
        "Min": float(vec.min()),
        "Max": float(vec.max()),
    })

norm_summary_df = pd.DataFrame(norm_summary_rows)
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 200):
    print(norm_summary_df.to_string(index=False))

## §2 — Survey: histograms, ΔBIC + Ashman's D, and annotated display

The survey has four parts. §2a shows raw density histograms for all 7
populations on a 3×3 grid grouped by phrase type so distributional shapes
can be compared visually. §2b applies the two-criterion bimodality test
formally (ΔBIC > 10 AND Ashman's D > 2) — a number, not a judgment call.
§2c confirms that full and validation splits agree on shape parameters,
justifying showing only the full version of each phrase type in the
annotated figure. §2d is the annotated display: a 3×2 figure with GMM
overlays on bimodal panels and mean lines on unimodal panels.

### §2a — Raw KDE curves (3×3 grid grouped by phrase type)

Density-normalized KDE curves for all 7 populations, organized by phrase
type (rows) × population variant (columns). Rows correspond to f_clue,
f_wndef, and f_wnex; columns correspond to *full*, *val*, and
*wnex-aligned subset*. The wnex-aligned column shows the same wnex word
population viewed through different phrase constructions — f_clue's
definitions restricted to wnex words, f_wndef's wnex-word subset, and
f_wnex itself (which is already wnex-only, sitting at the bottom of
the column directly below the two controls).

All panels share the same x-axis range (global min/max across all 7
norm vectors) so distributional shape is directly comparable. KDE curves
replace the earlier histograms because smooth densities make subtle mode
structure easier to see and compare across panels at this grid size.
Visual style: diagonal hatching for L2 (per FIGURE_STANDARDS.md), edge
color matching fill color, alpha=0.45, and a faint grid on every panel.
No GMM overlays yet — this figure is for visual inspection before the
formal analysis.


In [ ]:
# === §2a raw KDE: 3 rows (phrase type) x 3 columns (variant)
# Layout (None = empty cell, hidden):
#                    full              val              wnex-aligned
#   row 1  f_clue:   f_clue (full)     f_clue (val)     f_clue, def ∈ wnex
#   row 2  f_wndef:  f_wndef (full)    f_wndef (val)    f_wndef, wnex words
#   row 3  f_wnex:   None              None             f_wnex (full)
from scipy.stats import gaussian_kde

raw_layout = [
    ["f_clue (full)",  "f_clue (val)",  "f_clue, def ∈ wnex"],
    ["f_wndef (full)", "f_wndef (val)", "f_wndef, wnex words"],
    [None,             None,            "f_wnex (full)"],
]

# Shared x-range across all 7 visible panels so distributional shape is
# directly comparable. Computed once across all norm vectors, then applied
# explicitly to every visible panel — no `sharex=True`, since shared-axis
# behaviour interacts awkwardly with hidden panels in the layout (the
# f_wnex panel was previously rendering with a different x-tick range).
global_min = min(vec.min() for vec in norms.values())
global_max = max(vec.max() for vec in norms.values())
pad = 0.02 * (global_max - global_min)
xmin = global_min - pad
xmax = global_max + pad
# Backward-compatible aliases — §2d also uses these to set its xlim so the
# annotated display in §2d matches the raw KDE grid panel-for-panel.
x_lo, x_hi = xmin, xmax

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fill_color = MODEL_COLORS["g_stock"]

for r in range(3):
    for c in range(3):
        ax = axes[r, c]
        label = raw_layout[r][c]
        if label is None:
            # Hide unused cells in row 3 (left and middle).
            ax.set_visible(False)
            continue
        vec = norms[label]
        # KDE replaces the previous histogram bars: smoother visualization of
        # the same density. Hatched per FIGURE_STANDARDS (L2 = "//"); edge
        # color matches fill so the curve reads as a single shape.
        kde = gaussian_kde(vec)
        x_grid = np.linspace(xmin, xmax, 500)
        y = kde(x_grid)
        ax.fill_between(x_grid, y, alpha=0.45, color=fill_color,
                        hatch="//", edgecolor=fill_color)
        ax.set_title(label)
        ax.set_xlabel("L2 norm")
        ax.set_ylabel("density")
        # Apply the global x-range to every visible panel — no sharex coupling.
        ax.set_xlim(xmin, xmax)
        ax.grid(alpha=0.3)
        ax.annotate(f"N = {len(vec):,}",
                    xy=(0.97, 0.93), xycoords="axes fraction",
                    ha="right", va="top", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec="none", alpha=0.8))

fig.suptitle("g_stock L2 norm distributions by phrase type", fontsize=13)
fig.tight_layout()

raw_fig_path = FIG_DIR / "norm_bimodality_survey_raw.png"
fig.savefig(raw_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {raw_fig_path}")


### §2b — ΔBIC and Ashman's D table

For each of the 7 populations, fit a 1-component and a 2-component
Gaussian mixture and compute two complementary criteria. ΔBIC =
BIC(1) − BIC(2) measures whether the data statistically prefer the
2-component model — it grows with sample size, so at large N it can
flag any minor deviation from normality. Ashman's D measures how cleanly
the two component means are separated
(D = √2·|μ₂ − μ₁|/√(σ₁² + σ₂²)) and is independent of N. The full table
reports both criteria and the fitted GMM parameters; the focused table
classifies a population as bimodal only when **both** ΔBIC > 10 (Kass &
Raftery 1995, "very strong" evidence) **and** D > 2 (Ashman, Bird &
Zepf 1994, "cleanly separated" components).

In [ ]:
# === §2b compute ΔBIC and Ashman's D for every population
delta_bic_rows = []
for label, vec in norms.items():
    t0 = time.time()
    res = compute_delta_bic(vec)
    delta_bic_rows.append({
        "Population": label,
        "N": len(vec),
        "BIC(1)": res["bic_1"],
        "BIC(2)": res["bic_2"],
        "ΔBIC": res["delta_bic"],
        "Ashman's D": res["ashmans_d"],
        "μ1": res["mu1"], "σ1": res["sigma1"], "π1": res["weight1"],
        "μ2": res["mu2"], "σ2": res["sigma2"], "π2": res["weight2"],
    })
    print(f"{label:25s} ΔBIC={res['delta_bic']:>10.1f}  "
          f"D={res['ashmans_d']:>5.2f}  "
          f"μ1={res['mu1']:.3f}  μ2={res['mu2']:.3f}  "
          f"({time.time() - t0:.1f}s)")

# Sort most-bimodal first so the focused table reads naturally top-to-bottom.
delta_bic_df = (pd.DataFrame(delta_bic_rows)
                .sort_values("ΔBIC", ascending=False)
                .reset_index(drop=True))

In [ ]:
# === §2b full ΔBIC + Ashman's D table
print("=== Full ΔBIC table (sorted by ΔBIC desc) ===")
# Custom column-by-column formatting: BIC values use .1f (their magnitudes
# are in the thousands), ΔBIC uses .1f, Ashman's D uses .2f, GMM parameters
# use .4f.
disp = delta_bic_df.copy()
for c in ["BIC(1)", "BIC(2)", "ΔBIC"]:
    disp[c] = disp[c].map("{:.1f}".format)
disp["Ashman's D"] = disp["Ashman's D"].map("{:.2f}".format)
for c in ["μ1", "σ1", "π1", "μ2", "σ2", "π2"]:
    disp[c] = disp[c].map("{:.4f}".format)

# Reorder columns so Ashman's D sits immediately after ΔBIC for easy
# side-by-side comparison of the two bimodality criteria.
col_order = ["Population", "N", "BIC(1)", "BIC(2)", "ΔBIC", "Ashman's D",
             "μ1", "σ1", "π1", "μ2", "σ2", "π2"]
disp = disp[col_order]
with pd.option_context("display.width", 240, "display.max_colwidth", 60):
    print(disp.to_string(index=False))

In [ ]:
# === §2b focused bimodality view (two-criterion test)
DELTA_BIC_THRESHOLD = 10.0   # Kass & Raftery: "very strong" evidence
ASHMANS_D_THRESHOLD = 2.0    # Ashman, Bird & Zepf: "cleanly separated"

bimodal_view_df = delta_bic_df[
    ["Population", "N", "ΔBIC", "Ashman's D"]
].copy()

# Two-criterion test: a population is bimodal only when ΔBIC indicates the
# 2-component model is statistically preferred AND Ashman's D indicates the
# components are meaningfully separated. ΔBIC alone over-flags at large N.
bimodal_view_df["Bimodal?"] = [
    "Yes" if row["ΔBIC"] > DELTA_BIC_THRESHOLD
            and row["Ashman's D"] > ASHMANS_D_THRESHOLD
    else "No"
    for _, row in bimodal_view_df.iterrows()
]

print(f"=== Bimodality decisions "
      f"(ΔBIC > {DELTA_BIC_THRESHOLD:g} AND D > {ASHMANS_D_THRESHOLD:g}) ===")
disp = bimodal_view_df.copy()
disp["ΔBIC"] = disp["ΔBIC"].map("{:.1f}".format)
disp["Ashman's D"] = disp["Ashman's D"].map("{:.2f}".format)
with pd.option_context("display.width", 200):
    print(disp.to_string(index=False))

print("\nBimodal criteria: ΔBIC > 10 (Kass & Raftery 1995) AND "
      "Ashman's D > 2 (Ashman, Bird & Zepf 1994).")

# Capture labels for §2d using the two-criterion test — preserve the
# ΔBIC-descending order so the GMM overlay panels appear most-bimodal-first.
bimodal_mask = (
    (delta_bic_df["ΔBIC"] > DELTA_BIC_THRESHOLD)
    & (delta_bic_df["Ashman's D"] > ASHMANS_D_THRESHOLD)
)
bimodal_labels = delta_bic_df.loc[bimodal_mask, "Population"].tolist()

print(f"\n{len(bimodal_labels)} of {len(norms)} populations satisfy both "
      f"criteria:")
for lbl in bimodal_labels:
    print(f"  - {lbl}")

### §2c — Full ≈ val equivalence check

Before showing the annotated display figure, confirm that the
validation slices have the same distributional shape as their full-
dataset counterparts. ΔBIC scales with N, so it is not informative for
equivalence here; the relevant metrics are the N-independent ones —
Ashman's D and the GMM component parameters (μ₁, σ₁, μ₂, σ₂). If full
and val agree on these, the annotated display in §2d only needs to show
the full-dataset version of each phrase type.

In [ ]:
# === §2c full vs val side-by-side (shape parameters only)
fullval_pairs = [
    ("f_clue",  "f_clue (full)",  "f_clue (val)"),
    ("f_wndef", "f_wndef (full)", "f_wndef (val)"),
]
by_label = delta_bic_df.set_index("Population").to_dict(orient="index")

fullval_rows = []
for phrase_type, full_lbl, val_lbl in fullval_pairs:
    for split_name, lbl in [("full", full_lbl), ("val", val_lbl)]:
        row = by_label[lbl]
        fullval_rows.append({
            "Phrase type": phrase_type,
            "Split": split_name,
            "N": int(row["N"]),
            "D": row["Ashman's D"],
            "μ1": row["μ1"], "σ1": row["σ1"],
            "μ2": row["μ2"], "σ2": row["σ2"],
        })
fullval_df = pd.DataFrame(fullval_rows)

print("=== Full vs val shape-parameter comparison ===")
disp = fullval_df.copy()
disp["N"] = disp["N"].map("{:,}".format)
disp["D"] = disp["D"].map("{:.2f}".format)
for c in ["μ1", "σ1", "μ2", "σ2"]:
    disp[c] = disp[c].map("{:.4f}".format)
with pd.option_context("display.width", 200):
    print(disp.to_string(index=False))

print("\nSince full and val agree on Ashman's D and on the GMM component\n"
      "parameters (μ1, σ1, μ2, σ2), the annotated display figure in §2d\n"
      "shows only the full-dataset version of each phrase type — the val\n"
      "slice is distributionally redundant.")

### §2d — Annotated display figure (3×2 grid)

The annotated counterpart to §2a, structured for reading the bimodality
survey at a glance. Three rows correspond to the three phrase types
(f_clue, f_wndef, f_wnex). The left column shows the full-dataset
histogram for each phrase type; the right column shows the wnex-aligned
subset (def ∈ wnex for f_clue, wnex words for f_wndef, and f_wnex itself
in row 3). Row 3 left is empty — there is no "full" companion to
f_wnex distinct from f_wnex itself.

Each panel adds an overlay consistent with its bimodality classification
from §2b:

- **Bimodal panels** (ΔBIC > 10 AND D > 2): the two component PDFs as
  dashed black curves, the combined PDF as a solid black curve, dotted
  vertical lines at each component mean (annotated with μ values), and
  an annotation block reporting N, ΔBIC, and D.
- **Unimodal panels**: a single dashed vertical line at the population
  mean (annotated with μ), and an annotation block reporting N and D so
  the reader can see why it didn't qualify.

Visual style matches §2a: hatched bars (L2), edge color matching fill
color, alpha=0.45, faint grid, shared x-axis.

In [ ]:
# === §2d annotated 3x2 display figure
# Row layout (full | wnex-aligned). Row 3 left is empty (hidden).
annotated_layout = [
    ["f_clue (full)",  "f_clue, def ∈ wnex"],
    ["f_wndef (full)", "f_wndef, wnex words"],
    [None,             "f_wnex (full)"],
]

fit_by_label = delta_bic_df.set_index("Population").to_dict(orient="index")

fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True)
fill_color = MODEL_COLORS["g_stock"]

for r in range(3):
    for c in range(2):
        ax = axes[r, c]
        label = annotated_layout[r][c]
        if label is None:
            ax.set_visible(False)
            continue
        vec = norms[label]
        fit = fit_by_label[label]
        is_bimodal = label in bimodal_labels

        # KDE fill replaces histogram bars; same hatched/edge style as §2a so
        # the figures read as a pair (§2a unannotated, §2d adds GMM overlays).
        # gaussian_kde produces a probability density that integrates to 1,
        # matching the GMM component PDFs that are overlaid below — no
        # rescaling needed.
        kde = gaussian_kde(vec)
        x_grid = np.linspace(x_lo, x_hi, 500)
        y_kde = kde(x_grid)
        ax.fill_between(x_grid, y_kde, alpha=0.45, color=fill_color,
                        hatch="//", edgecolor=fill_color)
        ax.set_title(label)
        ax.set_xlabel("L2 norm")
        ax.set_ylabel("density")
        ax.set_xlim(x_lo, x_hi)
        ax.grid(alpha=0.3)

        # Build the per-component PDFs / mean line over the population's
        # x-range; annotation text is assembled differently for the two
        # bimodality outcomes.
        if is_bimodal:
            x = np.linspace(float(vec.min()), float(vec.max()), 400)
            mu1, sigma1, w1 = fit["μ1"], fit["σ1"], fit["π1"]
            mu2, sigma2, w2 = fit["μ2"], fit["σ2"], fit["π2"]
            pdf1 = w1 * sp_norm.pdf(x, loc=mu1, scale=sigma1)
            pdf2 = w2 * sp_norm.pdf(x, loc=mu2, scale=sigma2)
            pdf_total = pdf1 + pdf2

            # Per-component PDFs (dashed) and combined PDF (solid).
            ax.plot(x, pdf1, color="black", linewidth=1.5, linestyle="--",
                    alpha=0.85)
            ax.plot(x, pdf2, color="black", linewidth=1.5, linestyle="--",
                    alpha=0.85)
            ax.plot(x, pdf_total, color="black", linewidth=1.8,
                    linestyle="-", alpha=0.6)

            # Mean lines + annotations. Stagger annotation y-position so
            # labels don't collide when the means are close.
            y_top = ax.get_ylim()[1]
            for i, mu in enumerate([mu1, mu2], start=1):
                ax.axvline(mu, color="black", linewidth=1, linestyle=":",
                           alpha=0.7)
                voffset = -0.06 * y_top if i == 2 else 0
                ax.annotate(f"μ={mu:.2f}",
                            xy=(mu, y_top * 0.92 + voffset),
                            fontsize=8, ha="center", color="black",
                            bbox=dict(boxstyle="round,pad=0.2", fc="white",
                                      ec="none", alpha=0.7))

            # Bimodal annotation block: N + ΔBIC + D.
            d_val = fit["Ashman's D"]
            ann_text = (f"N = {len(vec):,}\n"
                        f"ΔBIC = {fit['ΔBIC']:,.0f} · D = {d_val:.2f}")
        else:
            # Single dashed line at the population mean.
            mean_val = float(vec.mean())
            ax.axvline(mean_val, color="black", linewidth=1.2,
                       linestyle="--", alpha=0.8)
            y_top = ax.get_ylim()[1]
            ax.annotate(f"μ={mean_val:.2f}",
                        xy=(mean_val, y_top * 0.92),
                        fontsize=8, ha="center", color="black",
                        bbox=dict(boxstyle="round,pad=0.2", fc="white",
                                  ec="none", alpha=0.7))

            # Unimodal annotation block: N + D (showing why it didn't qualify).
            d_val = fit["Ashman's D"]
            ann_text = (f"N = {len(vec):,}\n"
                        f"D = {d_val:.2f} (unimodal)")

        ax.annotate(ann_text,
                    xy=(0.97, 0.78), xycoords="axes fraction",
                    ha="right", va="top", fontsize=9,
                    bbox=dict(boxstyle="round,pad=0.2", fc="white",
                              ec="none", alpha=0.85))

fig.suptitle("g_stock L2 norm bimodality survey", fontsize=13)
fig.tight_layout()

annotated_fig_path = FIG_DIR / "norm_bimodality_survey_gstock.png"
fig.savefig(annotated_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {annotated_fig_path}")


### §2e — 3-component GMM diagnostic (f_wndef, wnex words)

A targeted diagnostic on `f_wndef, wnex words` only: does forcing a third
component meaningfully improve the BIC over the 2-component fit, or does
the 2-component model already capture the structure of this distribution?


In [ ]:
# === §2e 3-component GMM diagnostic on f_wndef, wnex words
# Targeted single-population diagnostic: extend the §2b 2-component fit to a
# 3-component fit and compare BICs. Lower BIC = preferred. If ΔBIC vs the
# 2-component fit is small (or negative), 2 components already explain the
# structure adequately.

target_label = "f_wndef, wnex words"
target_vec = norms[target_label]
X_target = target_vec.reshape(-1, 1)

# Fit 1, 2, 3 components with the same random_state as everywhere else so
# results are reproducible and directly comparable to §2b.
gmm_fits = {}
bic_table_rows = []
for k in (1, 2, 3):
    gm_k = GaussianMixture(n_components=k, random_state=RANDOM_STATE).fit(X_target)
    gmm_fits[k] = gm_k
    bic_table_rows.append({"n_components": k, "BIC": gm_k.bic(X_target)})

# ΔBIC computed against the 1-component fit so all entries share a baseline.
bic_1 = bic_table_rows[0]["BIC"]
for row in bic_table_rows:
    row["ΔBIC vs 1-component"] = bic_1 - row["BIC"]

bic_df = pd.DataFrame(bic_table_rows)
print(f"=== 3-component GMM BIC comparison: {target_label} ===")
print(bic_df.to_string(
    index=False,
    formatters={
        "BIC": "{:.1f}".format,
        "ΔBIC vs 1-component": "{:.1f}".format,
    },
))

# Extract the 3-component parameters and sort by mean so component indices
# are stable and the labels read low -> high.
gm3 = gmm_fits[3]
mus3 = gm3.means_.ravel()
sigmas3 = np.sqrt(gm3.covariances_.ravel())
weights3 = gm3.weights_
order3 = np.argsort(mus3)
mus3, sigmas3, weights3 = mus3[order3], sigmas3[order3], weights3[order3]

# Single-panel KDE + 3-component overlay. Visual style matches §2d:
# density-normalized KDE fill, hatched ("//") per FIGURE_STANDARDS, grid.
fig, ax = plt.subplots(figsize=(8, 5))
fill_color = MODEL_COLORS["g_stock"]
kde_target = gaussian_kde(target_vec)
x_kde = np.linspace(float(target_vec.min()), float(target_vec.max()), 500)
ax.fill_between(x_kde, kde_target(x_kde), alpha=0.45, color=fill_color,
                hatch="//", edgecolor=fill_color)

# Three contrasting non-blue colors — distinct from the blue KDE fill so
# the per-component PDFs are easy to tell apart from each other and from
# the data.
component_colors = ["#d62728", "#2ca02c", "#9467bd"]  # red, green, purple

x_grid = np.linspace(float(target_vec.min()), float(target_vec.max()), 400)
pdf_total = np.zeros_like(x_grid)
for i, (mu, sigma, w, col) in enumerate(zip(mus3, sigmas3, weights3, component_colors)):
    pdf_i = w * sp_norm.pdf(x_grid, loc=mu, scale=sigma)
    pdf_total += pdf_i
    ax.plot(x_grid, pdf_i, color=col, linewidth=1.5, linestyle="--",
            alpha=0.9, label=f"comp {i+1} (μ={mu:.2f}, π={w:.2f})")

# Combined PDF in solid black, slightly transparent so the per-component
# dashes remain visible underneath.
ax.plot(x_grid, pdf_total, color="black", linewidth=1.8, linestyle="-",
        alpha=0.6, label="combined PDF")

# Vertical dotted line at each component mean with a μ annotation in a
# white-background box (matches §2d annotation style).
y_top = ax.get_ylim()[1]
for i, mu in enumerate(mus3):
    ax.axvline(mu, color="black", linewidth=1, linestyle=":", alpha=0.7)
    # Stagger y-positions so labels don't collide when means are close.
    voffset = -0.06 * y_top * (i % 2)
    ax.annotate(f"μ={mu:.2f}",
                xy=(mu, y_top * 0.92 + voffset),
                fontsize=8, ha="center", color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white",
                          ec="none", alpha=0.7))

ax.set_xlabel("L2 norm")
ax.set_ylabel("density")
ax.set_title(f"{target_label} — 3-component GMM diagnostic")
ax.grid(alpha=0.3)
ax.legend(loc="upper left", fontsize=8)

fig.tight_layout()
plt.show()
# Diagnostic figure only — not saved to disk.



## §3 — Is L2 norm a property of the word?

The bimodality in f_clue norms raises a natural question: is L2 norm a
stable property of the word being embedded, or is it determined by the
surrounding context? If the same word consistently lands at a similar
norm regardless of which clue it appears in, the bimodality reflects
something about *certain words*. If the same word's norm varies widely
across clues, the bimodality is driven by the *interaction between the
word and its context*. Two complementary analyses test this.


### §3a — Intraclass correlation (ICC) on f_clue norms

Each definition word appears in multiple clues, giving it multiple f_clue
norms. The intraclass correlation coefficient (ICC) quantifies what
fraction of total norm variance is *between-word* (driven by word
identity) versus *within-word* (driven by clue context). We use ICC(1,1),
the one-way random-effects formulation:

    ICC = (MS_between − MS_within) / (MS_between + (k − 1) · MS_within)

where MS_between is the mean square between groups (words), MS_within is
the mean square within groups (clues of the same word), and k is the
average group size. A high ICC means norms are mostly a property of the
word; a low ICC means norms vary so much across clues for the same word
that word identity barely constrains them.

We then visualize the same finding directly. For ~25 definition words
that each appear in at least 30 clues — sampled across the f_clue norm
range so the plot includes words from both modes — we draw a vertical
strip of all that word's f_clue norms. Layered on top: the word's
decontextualized f_wndef norm (diamond) and, where available, its
f_wnex norm (triangle). Horizontal reference lines at the two GMM
component means from §2 mark where the f_clue modes sit. If ICC is low,
most words' strips will span both modes, showing that individual words
do not have a characteristic norm.

Each f_clue dot is additionally color-coded by the position of the
definition within the surface: **start** (orange, `#E69F00` — surface
begins with the definition), **middle** (bluish-green, `#009E73` —
neither start nor end), or **end** (reddish-purple, `#CC79A7` — surface
ends with the definition). When a clue's surface is exactly the
definition (so it would qualify as both start and end), **start wins**
under a deterministic precedence rule. The Okabe-Ito qualitative palette
is used because it is colorblind-friendly and visually distinct from the
white-fill diamond/triangle markers and from the g_stock blue used
elsewhere in the notebook. This encoding lets the reader see whether
each word's mode-spanning behavior is associated with a particular
syntactic role of the definition in the clue.


In [ ]:
# === §3a ICC(1,1) on f_clue norms + strip plot
# Build a per-clue table joining f_clue_index against clues_wn_filtered to
# attach the WordNet-normalized definition (definition_wn) to every f_clue
# row, then attach the L2 norm from norms["f_clue (full)"] via the row index.
fclue_word_norms = merged.assign(
    norm=norms["f_clue (full)"][merged["row"].values]
)[["definition_wn", "norm"]]

# Filter to definition words with >= 2 appearances — ICC is undefined for
# singletons (within-group variance has no degrees of freedom).
group_sizes = fclue_word_norms.groupby("definition_wn").size()
multi_words = group_sizes[group_sizes >= 2].index
icc_data = fclue_word_norms[fclue_word_norms["definition_wn"].isin(multi_words)]

n_words = icc_data["definition_wn"].nunique()
n_rows = len(icc_data)
print(f"Words with >=2 f_clue appearances: {n_words:,}")
print(f"Clue rows covered:                  {n_rows:,}  "
      f"(of {len(fclue_word_norms):,} total f_clue rows)")

# One-way ANOVA decomposition for ICC(1,1).
#   SS_total   = sum_{ij} (x_ij - grand_mean)^2
#   SS_between = sum_i n_i (group_mean_i - grand_mean)^2
#   SS_within  = sum_{ij} (x_ij - group_mean_i)^2
# Degrees of freedom: between = (n_groups - 1), within = (N - n_groups).
grouped = icc_data.groupby("definition_wn")["norm"]
group_means = grouped.mean()
group_n     = grouped.size()
grand_mean  = icc_data["norm"].mean()
N_total     = len(icc_data)
n_groups    = len(group_means)

ss_between = float((group_n * (group_means - grand_mean) ** 2).sum())
ss_within  = float(((icc_data["norm"] - icc_data["definition_wn"].map(group_means)) ** 2).sum())

df_between = n_groups - 1
df_within  = N_total - n_groups
ms_between = ss_between / df_between
ms_within  = ss_within  / df_within

# Average group size (for unbalanced designs, ICC(1,1) uses the mean n_i).
k_bar = float(group_n.mean())

icc_value = (ms_between - ms_within) / (ms_between + (k_bar - 1) * ms_within)
print(f"\nICC(1,1):              {icc_value:.4f}")
print(f"  MS_between:          {ms_between:.4f}")
print(f"  MS_within:           {ms_within:.4f}")
print(f"  Avg group size (k):  {k_bar:.2f}")

# Interpretation guide (Koo & Li 2016 style thresholds).
if icc_value < 0.05:
    interp = "negligible word-level consistency"
elif icc_value < 0.20:
    interp = "weak word-level consistency"
elif icc_value < 0.50:
    interp = "moderate word-level consistency"
else:
    interp = "strong word-level consistency"
print(f"  Interpretation:      {interp}")

# Within-word vs population spread — a second angle on the same question.
within_word_std = grouped.std(ddof=1).dropna()
median_within_std = float(within_word_std.median())
overall_std       = float(icc_data["norm"].std(ddof=1))
print(f"\nMedian within-word std: {median_within_std:.4f}")
print(f"Overall population std: {overall_std:.4f}")
print(f"Ratio (within / overall): {median_within_std / overall_std:.3f}  "
      "— close to 1 means individual words spread almost as widely as the whole population")

# Build a per-row f_clue table that carries `surface` and `definition_wn`
# alongside the L2 norm and a categorical `position` label classifying where
# the definition sits in the surface (start / middle / end). This frame is
# computed once and then consumed by both the strip plot below (color
# encoding) and the example-clues cell that follows. Pulling `surface` here
# avoids re-merging f_clue_index against clues_wn_filtered later on.
fclue_surfaces = f_clue_index.merge(
    clues_wn[["clue_id", "definition", "definition_wn", "surface"]],
    on=["clue_id", "definition"],
)
assert len(fclue_surfaces) == len(f_clue_index), (
    f"Expected 1:1 join; got {len(fclue_surfaces)} rows from {len(f_clue_index)}"
)
fclue_surfaces = fclue_surfaces.assign(
    norm=norms["f_clue (full)"][fclue_surfaces["row"].values]
)

# Position classification: lowercased prefix/suffix match. A clue whose
# surface IS the definition would satisfy both start and end; we pick a
# deterministic precedence in which START WINS over END so the categories
# are mutually exclusive and reproducible.
surface_lower = fclue_surfaces["surface"].str.lower()
def_lower     = fclue_surfaces["definition"].str.lower()
starts = surface_lower.values == def_lower.values  # placeholder, overwritten below
# Use vectorized startswith/endswith on the Series elementwise. There is no
# pandas built-in for "startswith using another column", so a list
# comprehension is the clean way to do this across ~239K rows.
starts = np.array([s.startswith(d) for s, d in zip(surface_lower, def_lower)])
ends   = np.array([s.endswith(d)   for s, d in zip(surface_lower, def_lower)])

# start precedence: classify as "start" if starts is True, else "end" if
# ends is True, else "middle". Documented in the §3a markdown cell above.
position = np.where(starts, "start",
            np.where(ends, "end", "middle"))
fclue_surfaces["position"] = position

# Quick sanity counts so the notebook reader can see the breakdown.
print("\nDefinition position in surface (across all 239,406 f_clue rows):")
print(fclue_surfaces["position"].value_counts().to_string())


In [ ]:
# === §3a strip plot: ~25 words spanning the f_clue norm range
# Pick words appearing in >=30 clues so each strip has dense vertical
# coverage and a stable mean. Then sample evenly across the per-word mean
# norm range so the plot includes words from both modes — not just the
# most frequent ones, which would cluster around the population mean.
strip_min_clues = 30
strip_n_words   = 25

# Lookup dicts for the decontextualized markers we layer on each strip:
# f_wndef value (every word that has a wndef row) and f_wnex value (only
# the 8,360 words in the wnex vocabulary).
f_wndef_norm_by_word = dict(zip(vocab_wndef["word"],
                                norms["f_wndef (full)"]))
f_wnex_norm_by_word  = dict(zip(vocab_wnex["word"],
                                norms["f_wnex (full)"]))

eligible_sizes = group_sizes[group_sizes >= strip_min_clues]
eligible_means = (
    fclue_word_norms[fclue_word_norms["definition_wn"].isin(eligible_sizes.index)]
    .groupby("definition_wn")["norm"].mean()
    .sort_values()
)
print(f"Eligible words (>={strip_min_clues} clues): {len(eligible_means):,}")

# Quantile sampling: split the sorted-by-mean-norm word list into N bins of
# equal cardinality, then take one word per bin (the bin's middle word) to
# guarantee spread without relying on RNG.
n_eligible = len(eligible_means)
bin_idx = np.linspace(0, n_eligible - 1, strip_n_words).round().astype(int)
strip_words = eligible_means.iloc[bin_idx].index.tolist()
print(f"Selected {len(strip_words)} strip words spanning "
      f"f_clue mean norm {eligible_means.iloc[bin_idx[0]]:.2f}"
      f" – {eligible_means.iloc[bin_idx[-1]]:.2f}")

# Okabe-Ito colorblind-friendly qualitative palette for the three position
# categories. These three colors are visually distinct from each other,
# from the white-fill black-edge diamond/triangle markers used for f_wndef
# and f_wnex, and from MODEL_COLORS["g_stock"] (#1f77b4). The g_stock blue
# is intentionally avoided here because it previously encoded "f_clue dot"
# generically — we are now subdividing those dots by position.
POSITION_COLORS = {
    "start":  "#E69F00",  # Okabe-Ito orange
    "middle": "#009E73",  # Okabe-Ito bluish-green
    "end":    "#CC79A7",  # Okabe-Ito reddish-purple
}

# For each selected strip word, gather its per-clue f_clue norms together
# with the position label so dots can be colored. Pull from fclue_surfaces
# (built in cell above) which already carries norm and position.
strip_rows = []
for w in strip_words:
    word_rows = fclue_surfaces[fclue_surfaces["definition_wn"] == w]
    fclue_norms_w   = word_rows["norm"].values
    fclue_position_w = word_rows["position"].values
    fwndef = f_wndef_norm_by_word.get(w, np.nan)
    fwnex  = f_wnex_norm_by_word.get(w, np.nan)  # NaN when word not in wnex
    strip_rows.append({
        "word":       w,
        "norms":      fclue_norms_w,
        "positions":  fclue_position_w,
        "mean":       float(fclue_norms_w.mean()),
        "n_clues":    len(fclue_norms_w),
        "f_wndef":    fwndef,
        "f_wnex":     fwnex,
    })

# Sort by mean f_clue norm so the x-axis reads left-to-right low-to-high.
strip_rows.sort(key=lambda r: r["mean"])

# Tally the position breakdown across the highlighted words so the reader
# knows what the colors represent in this specific plot.
strip_pos_counts = pd.Series(
    np.concatenate([r["positions"] for r in strip_rows])
).value_counts()
print("Position breakdown across the {} highlighted words:".format(len(strip_rows)))
print(strip_pos_counts.to_string())

# Pull the f_clue (full) GMM component means for the horizontal reference lines.
fclue_fit = fit_by_label["f_clue (full)"]
ref_mu1, ref_mu2 = fclue_fit["μ1"], fclue_fit["μ2"]

fig, ax = plt.subplots(figsize=(14, 5))

x_positions = np.arange(len(strip_rows))

# Reference lines at the two f_clue GMM component means — show where the
# bimodal modes sit so the reader can see whether each word's strip
# crosses both modes.
ax.axhline(ref_mu1, color="0.55", linestyle="--", linewidth=1, alpha=0.8,
           zorder=0)
ax.axhline(ref_mu2, color="0.55", linestyle="--", linewidth=1, alpha=0.8,
           zorder=0)
ax.text(len(strip_rows) - 0.5, ref_mu1, f" μ₁={ref_mu1:.2f}",
        fontsize=8, color="0.4", va="center", ha="left")
ax.text(len(strip_rows) - 0.5, ref_mu2, f" μ₂={ref_mu2:.2f}",
        fontsize=8, color="0.4", va="center", ha="left")

# Per-word strips with a small horizontal x-jitter so overplotted dots are
# distinguishable; alpha=0.4 handles overplotting on words with many clues.
# Dot color encodes the definition's syntactic position in the surface.
rng_strip = np.random.default_rng(seed=RANDOM_STATE)
for i, row in enumerate(strip_rows):
    n = len(row["norms"])
    jitter = rng_strip.uniform(-0.18, 0.18, size=n)
    # Map each dot's position label to a color via POSITION_COLORS so the
    # entire batch can be drawn in a single scatter call (faster than per-
    # category sub-scatters and gives matplotlib a homogeneous Nx3 array).
    dot_colors = [POSITION_COLORS[p] for p in row["positions"]]
    ax.scatter(np.full(n, i) + jitter, row["norms"],
               s=10, c=dot_colors, alpha=0.4, edgecolor="none",
               zorder=2)
    # Per-word mean as a short black horizontal bar.
    ax.plot([i - 0.32, i + 0.32], [row["mean"], row["mean"]],
            color="black", linewidth=2, zorder=3)
    # f_wndef diamond (every word has one — every selected word came from
    # eligible_means, which uses words seen as f_clue definitions, but only
    # words in the wndef vocab will have a f_wndef value; in practice all
    # f_clue defs are in wndef by construction).
    if not np.isnan(row["f_wndef"]):
        ax.scatter([i], [row["f_wndef"]], marker="D", s=55,
                   color="white", edgecolor="black", linewidth=1.2,
                   zorder=4)
    # f_wnex triangle — only present for words in the wnex vocabulary.
    if not np.isnan(row["f_wnex"]):
        ax.scatter([i], [row["f_wnex"]], marker="^", s=55,
                   color="white", edgecolor="black", linewidth=1.2,
                   zorder=4)

ax.set_xticks(x_positions)
ax.set_xticklabels([r["word"] for r in strip_rows], rotation=60,
                   ha="right", fontsize=9)
ax.set_xlabel("definition word (sorted by mean f_clue norm)")
ax.set_ylabel("L2 norm")
ax.set_title("Per-word f_clue norm strips, with decontextualized f_wndef / f_wnex markers")
ax.grid(alpha=0.3, axis="y")

# Legend handles built manually so dot color (one per position category),
# diamond, and triangle each map cleanly to phrase types and definition
# positions. Three position entries replace the previous single "f_clue
# norm" entry, since the dots are now sub-coded by position.
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor=POSITION_COLORS["start"],
           alpha=0.8, markersize=7,
           label="f_clue dot — definition at surface start"),
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor=POSITION_COLORS["middle"],
           alpha=0.8, markersize=7,
           label="f_clue dot — definition in middle"),
    Line2D([0], [0], marker="o", color="none",
           markerfacecolor=POSITION_COLORS["end"],
           alpha=0.8, markersize=7,
           label="f_clue dot — definition at surface end"),
    Line2D([0], [0], color="black", linewidth=2,
           label="mean f_clue norm"),
    Line2D([0], [0], marker="D", color="none", markerfacecolor="white",
           markeredgecolor="black", markersize=8, label="f_wndef norm"),
    Line2D([0], [0], marker="^", color="none", markerfacecolor="white",
           markeredgecolor="black", markersize=8,
           label="f_wnex norm (where available)"),
    Line2D([0], [0], color="0.55", linestyle="--", linewidth=1,
           label="f_clue GMM component means"),
]
# bbox_to_anchor lifts the legend into the upper-right outside-plot region
# so it never obscures the rightmost word strip's dots or markers.
ax.legend(handles=legend_handles, loc="upper left",
          bbox_to_anchor=(1.01, 1.0), fontsize=8, framealpha=0.9,
          borderaxespad=0.0)

fig.tight_layout()

icc_strip_path = FIG_DIR / "norm_bimodality_icc_strip.png"
fig.savefig(icc_strip_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {icc_strip_path}")


### §3a — Example clues from each f_clue norm mode

The strip plot above shows that some words have f_clue norms spanning both
GMM modes — the same word can land near the lower mode in one clue and the
upper mode in another. To make this concrete, the cell below pulls actual
clue surfaces for three such words ("european", "style", "song") and prints
three examples from each side of the ~30.5 valley between the two modes.
The split is purely illustrative: no mode assignment is being made here, and
the 30.5 cutoff is just the approximate trough between the two GMM
components (μ₁ ≈ 29.6, μ₂ ≈ 31.6) from §2.

The three picks per side are drawn from the within-mode 25th, 50th, and 75th
percentile so the examples span each mode's range rather than just its
extremes.

In [ ]:
# === §3a example clues from each f_clue norm mode for three words
# fclue_surfaces was built in the ICC cell above and already carries
# `surface`, `definition_wn`, `norm`, and `position` — no rebuild needed.

# Approximate valley between the two f_clue GMM components — used only as
# a convenient illustrative split, NOT as a mode assignment.
MODE_SPLIT = 30.5
EXAMPLE_WORDS = ["european", "style", "song"]
N_PER_MODE = 3

def pick_quantile_examples(df_mode, n=N_PER_MODE):
    """Pick rows near the 25th, 50th, 75th percentile of `norm` within a mode.

    Sort by norm, then take the rows at integer positions corresponding to
    those quantiles. Falls back to taking whatever is available if the mode
    has fewer than n rows.
    """
    df_sorted = df_mode.sort_values("norm").reset_index(drop=True)
    n_rows = len(df_sorted)
    if n_rows == 0:
        return df_sorted
    if n_rows <= n:
        return df_sorted
    # Integer positions for the 25th/50th/75th (or evenly spaced if n != 3).
    quantile_fracs = np.linspace(0.25, 0.75, n)
    idxs = np.clip(np.round(quantile_fracs * (n_rows - 1)).astype(int),
                   0, n_rows - 1)
    return df_sorted.iloc[idxs]

for word in EXAMPLE_WORDS:
    word_rows = fclue_surfaces[fclue_surfaces["definition_wn"] == word]
    lower = word_rows[word_rows["norm"] < MODE_SPLIT]
    upper = word_rows[word_rows["norm"] >= MODE_SPLIT]
    print(f"{word} ({len(word_rows)} clues: "
          f"{len(lower)} lower mode, {len(upper)} upper mode)")
    print()

    # Lower mode (norm < 30.5): three picks at the within-mode quantiles.
    print(f"  Lower mode (norm < {MODE_SPLIT}):")
    for _, r in pick_quantile_examples(lower).iterrows():
        print(f"    {r['norm']:.2f}  {r['surface']}")
    print()

    # Upper mode (norm >= 30.5): same quantile-based selection.
    print(f"  Upper mode (norm >= {MODE_SPLIT}):")
    for _, r in pick_quantile_examples(upper).iterrows():
        print(f"    {r['norm']:.2f}  {r['surface']}")
    print()


### §3b — Cross-format norm correlation

If L2 norm is a property of the word itself, the same word should land at
a similar norm regardless of which phrase format it is embedded in.
Restricting to the wnex vocabulary (so every word has both an f_wndef
phrase and an f_wnex phrase, and at least one f_clue appearance), we
compute Spearman ρ for all three pairwise comparisons:

- mean f_clue norm vs f_wndef norm
- mean f_clue norm vs f_wnex norm
- f_wndef norm vs f_wnex norm

Spearman ρ tests whether the word-by-word *ordering* on the norm axis is
preserved across phrase constructions. With thousands of words, the
magnitude of ρ is what matters, not its p-value. The two f_clue-vs-X
panels probe whether contextualized norm is anchored to a stable
word-level property; the third panel sets the ceiling — if even the two
decontextualized formats only weakly agree, no f_clue-vs-X correlation
can do better.

In [ ]:
# === §3b cross-format scatters: 1x3 panel for all three pairwise comparisons
from scipy.stats import spearmanr

# Restrict to the wnex vocabulary (8,360 words) — same subset used elsewhere
# for cross-format comparisons. For each wnex word, compute mean f_clue norm
# and look up its f_wndef and f_wnex norms via the existing dicts.
mean_fclue_norm_by_word = (
    fclue_word_norms.groupby("definition_wn")["norm"].mean()
)

cross_rows = []
for w in vocab_wnex["word"]:
    if w not in mean_fclue_norm_by_word.index:
        continue  # wnex word that never appears as a definition in f_clue
    if w not in wndef_word_to_row:
        continue  # wnex ⊆ wndef should hold, but guard against drift
    cross_rows.append({
        "word":           w,
        "mean_f_clue":    float(mean_fclue_norm_by_word[w]),
        "f_wndef":        float(norms["f_wndef (full)"][wndef_word_to_row[w]]),
        "f_wnex":         float(f_wnex_norm_by_word[w]),
    })
cross_df = pd.DataFrame(cross_rows)
N_cross = len(cross_df)
print(f"N words with f_clue, f_wndef, and f_wnex norms: {N_cross:,}  "
      f"(of {len(vocab_wnex):,} wnex words)")

# Three pairwise Spearman correlations — the comparison the panels visualize.
rho_fclue_wndef, _ = spearmanr(cross_df["f_wndef"], cross_df["mean_f_clue"])
rho_fclue_wnex,  _ = spearmanr(cross_df["f_wnex"],  cross_df["mean_f_clue"])
rho_wndef_wnex,  _ = spearmanr(cross_df["f_wnex"],  cross_df["f_wndef"])
print(f"Spearman ρ (mean f_clue vs f_wndef): {rho_fclue_wndef:.3f}")
print(f"Spearman ρ (mean f_clue vs f_wnex):  {rho_fclue_wnex:.3f}")
print(f"Spearman ρ (f_wndef vs f_wnex):      {rho_wndef_wnex:.3f}")

# Keep the f_clue-vs-f_wndef ρ under its old name so §5a/§5b reuse it
# without changes to those cells.
rho_cross = rho_fclue_wndef

# Panel definitions: x-column, y-column, x-label, y-label, ρ value, title.
# Title convention: "y vs x" (y first), per the spec.
panels = [
    ("f_wndef", "mean_f_clue",
     "f_wndef norm", "mean f_clue norm",
     rho_fclue_wndef,
     "mean f_clue norm vs f_wndef norm"),
    ("f_wnex",  "mean_f_clue",
     "f_wnex norm",  "mean f_clue norm",
     rho_fclue_wnex,
     "mean f_clue norm vs f_wnex norm"),
    ("f_wnex",  "f_wndef",
     "f_wnex norm",  "f_wndef norm",
     rho_wndef_wnex,
     "f_wndef norm vs f_wnex norm"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
scatter_color = MODEL_COLORS["g_stock"]

for ax, (xcol, ycol, xlab, ylab, rho, title) in zip(axes, panels):
    x = cross_df[xcol].values
    y = cross_df[ycol].values

    # Cloud: small low-alpha dots, FIGURE_STANDARDS color for g_stock.
    ax.scatter(x, y, s=12, color=scatter_color, alpha=0.3, edgecolor="none")

    # OLS best-fit line — Spearman is the reported statistic, but a
    # least-squares trend line gives the eye an anchor for the cloud's
    # overall direction.
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept,
            color="black", linewidth=1.5, linestyle="--", alpha=0.8)

    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.grid(alpha=0.3)

    # Annotation box (upper left): Spearman ρ and N.
    ax.annotate(f"Spearman ρ = {rho:.3f}\nN = {N_cross:,}",
                xy=(0.04, 0.96), xycoords="axes fraction",
                ha="left", va="top", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.3",
                          fc="white", ec="0.7", alpha=0.9))

fig.suptitle("Cross-format norm correlations (wnex vocabulary)",
             fontsize=12, y=1.02)
fig.tight_layout()

cross_fig_path = FIG_DIR / "norm_bimodality_cross_format_scatter.png"
fig.savefig(cross_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {cross_fig_path}")


## §4 — What surface features influence f_clue L2 norm?

The ICC and cross-format analysis establish that f_clue norm is primarily
contextual — the same word's norm varies widely across clues, and rank
ordering is barely preserved across phrase formats. This section asks
which surface-level properties of the f_clue input text predict the L2
norm of the output embedding. We extract six tokenizer-level features
(measured in **subword tokens** — what the transformer actually
processes — except `t_capitalized`, a flag on the target word's display
form), then join in nine binary wordplay flags from
`wordplay_metadata.csv` so the regression can ask whether surface
tokenization or wordplay structure (or both) drives the f_clue L2 norm.
The fitted OLS reports both a 6-feature R² and the full 15-feature R²,
making the incremental contribution of wordplay visible.


### §4a — Feature engineering

For every f_clue phrase we extract six surface features, then join in
nine binary wordplay flags from `wordplay_metadata.csv` (one per common
cryptic wordplay device). The full predictor set used by §4c's OLS is
therefore 15 features wide.

**Surface features (six):**

| Feature | Definition |
|---------|------------|
| `t_tokens` | Subword tokens in the target text (between `<t>` and `</t>`, excluding the delimiters themselves) |
| `p_tokens` | Total subword tokens in the full phrase, excluding `[CLS]`, `[SEP]`, and the `<t></t>` delimiters |
| `t_position` | Token offset of the `<t>` delimiter divided by total token count (0 = start, 1 = end) |
| `t_p_ratio` | `t_tokens / p_tokens` |
| `t_capitalized` | 1 if the first character of the target text is uppercase, 0 otherwise |
| `p_punctuation` | Number of standard punctuation tokens in the full phrase (delimiter `<`, `>`, `/`, `t` characters that comprise `<t></t>` are excluded) |

**Wordplay features (nine):** `double_def`, `anagram_single_word`,
`anagram_consec_words`, `hidden_fwd`, `hidden_rev`, `selection_firsts`,
`selection_lasts`, `selection_alt`, `selection_alt_rev`. These are the
binary flags from the structural-pattern detector covering the wordplay
devices in the f_clue notebook's analysis. The two
`selection_*_rev` variants of firsts/lasts are out of scope here.

**Tokenizer behavior:** MBERT (`bert-base-multilingual-cased`) does not
treat `<t>` and `</t>` as single tokens — they split into individual
characters (`<`, `t`, `>` and `<`, `/`, `t`, `>` respectively). Rather
than locating and excluding these inside the tagged-phrase tokenization,
we tokenize a *cleaned* phrase (with the `<t>` / `</t>` markers removed)
along with the target text on its own. This sidesteps any subword
artifacts at the delimiter boundaries and gives clean counts. The token
offset of `<t>` is then the length of the tokenization of the
left-of-target context.

This analysis uses the full f_clue dataset (239,406 rows) — the larger N
gives the multiple regression more leverage to separate the surface
features. Tokenizing 239K phrases takes ~1–2 minutes; we batch and print
progress.


In [ ]:
# === §4a feature engineering: tokenize 239K f_clue phrases
import re
import string
from transformers import AutoTokenizer

tokenizer_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
print(f"Loaded MBERT tokenizer in {time.time() - tokenizer_t0:.1f}s")

f_clue_phrases = pd.read_csv(
    WN_DIR / "clue_phrases" / "f_clue.csv",
    keep_default_na=False, na_values=[""],
)
print(f"f_clue.csv: {len(f_clue_phrases):,} rows  "
      f"(expected 239,406)")
assert len(f_clue_phrases) == 239_406

# Align f_clue phrases with the f_clue (full) norm vector by joining on
# (clue_id, definition). f_clue_index is the row order of f_clue.npy, and
# norms["f_clue (full)"] has those same rows.
f_clue_features_df = f_clue_index.merge(
    f_clue_phrases[["clue_id", "definition", "phrase"]],
    on=["clue_id", "definition"],
    how="left",
)
assert f_clue_features_df["phrase"].notna().all(), "Missing phrases after merge"
assert len(f_clue_features_df) == len(f_clue_index)

# Pre-attach the L2 norm so the resulting frame is self-contained.
f_clue_features_df["norm"] = norms["f_clue (full)"][f_clue_features_df["row"].values]

# Punctuation set used for p_punctuation. The delimiter characters '<', '>',
# '/' (which appear inside <t></t>) are dropped from this set so the
# delimiters themselves don't get counted — the spec calls for punctuation
# in the full phrase EXCLUDING the delimiters.
PUNCT_SET = set(string.punctuation) - {"<", ">", "/"}

# Pattern to peel <t>...</t> off the phrase. Use a non-greedy match for the
# target — short tagged spans, never nested.
TAG_PATTERN = re.compile(r"<t>(.*?)</t>", re.DOTALL)


def featurize_phrase(phrase):
    """Return (t_tokens, p_tokens, t_position, t_p_ratio, t_capitalized, p_punctuation).

    Strategy: peel off the <t>...</t> markers, tokenize three pieces
    independently — the target, the left context, and the cleaned full
    phrase — and combine. This avoids the subword artefacts that would
    arise from tokenizing across the delimiter boundary.
    """
    m = TAG_PATTERN.search(phrase)
    if m is None:
        # Should not happen for f_clue rows, but fail loud rather than guess.
        raise ValueError(f"No <t>...</t> tag in phrase: {phrase!r}")
    target = m.group(1)
    left_context = phrase[:m.start()]
    # Cleaned phrase: drop the <t> and </t> markers, keep all other text.
    clean_phrase = TAG_PATTERN.sub(lambda mm: mm.group(1), phrase)

    # add_special_tokens=False -> no [CLS]/[SEP], so counts are pure content.
    target_tokens = tokenizer.tokenize(target, add_special_tokens=False)
    clean_tokens  = tokenizer.tokenize(clean_phrase, add_special_tokens=False)
    left_tokens   = tokenizer.tokenize(left_context, add_special_tokens=False)

    t_tokens = len(target_tokens)
    p_tokens = len(clean_tokens)
    # Guard against zero-length cleaned phrase (shouldn't occur in real data).
    t_position = len(left_tokens) / p_tokens if p_tokens else 0.0
    t_p_ratio  = t_tokens / p_tokens if p_tokens else 0.0
    t_capitalized = int(bool(target) and target[0].isupper())
    p_punctuation = sum(1 for tok in clean_tokens if tok in PUNCT_SET)

    return (t_tokens, p_tokens, t_position, t_p_ratio,
            t_capitalized, p_punctuation)


# Process in chunks with progress prints — pure-Python tokenization at this
# volume runs in roughly 1–2 minutes on a laptop.
feat_t0 = time.time()
feat_arrays = {
    "t_tokens":      np.zeros(len(f_clue_features_df), dtype=np.int32),
    "p_tokens":      np.zeros(len(f_clue_features_df), dtype=np.int32),
    "t_position":    np.zeros(len(f_clue_features_df), dtype=np.float32),
    "t_p_ratio":     np.zeros(len(f_clue_features_df), dtype=np.float32),
    "t_capitalized": np.zeros(len(f_clue_features_df), dtype=np.int8),
    "p_punctuation": np.zeros(len(f_clue_features_df), dtype=np.int32),
}

phrases = f_clue_features_df["phrase"].values
report_every = 50_000
for i, phrase in enumerate(phrases):
    (feat_arrays["t_tokens"][i], feat_arrays["p_tokens"][i],
     feat_arrays["t_position"][i], feat_arrays["t_p_ratio"][i],
     feat_arrays["t_capitalized"][i], feat_arrays["p_punctuation"][i]) = (
        featurize_phrase(phrase)
    )
    if (i + 1) % report_every == 0:
        elapsed = time.time() - feat_t0
        rate = (i + 1) / elapsed
        eta = (len(phrases) - (i + 1)) / rate
        print(f"  tokenized {i + 1:>7,} / {len(phrases):,}  "
              f"({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

for col, arr in feat_arrays.items():
    f_clue_features_df[col] = arr
print(f"\nFeature extraction done in {time.time() - feat_t0:.1f}s")

# Quick sanity table — summary statistics for each feature plus the norm.
feat_cols = ["t_tokens", "p_tokens", "t_position", "t_p_ratio",
             "t_capitalized", "p_punctuation", "norm"]
print("\nFeature summary (mean / std / min / max):")
print(f_clue_features_df[feat_cols].agg(["mean", "std", "min", "max"])
      .T.to_string(float_format=lambda x: f"{x:.3f}"))


Wordplay flags come from `wordplay_metadata.csv`, the structural-pattern
detector's per-clue boolean output, and are joined onto
`f_clue_features_df` by `clue_id`. Multi-definition clues (multiple rows
per `clue_id` in `f_clue_features_df`) inherit the same flag for every
row — the wordplay flag describes the clue as a whole, which is the
intended behavior here.


In [ ]:
# === §4a wordplay metadata join: 9 binary wordplay features
# Boolean flags from the structural-pattern detector, one per common
# cryptic wordplay device. We left-join by clue_id and assert that no
# rows fall out and no NaNs are introduced — wordplay_metadata.csv
# covers the full clue corpus, so every f_clue clue_id should match.
wordplay = pd.read_csv(
    PROJECT_ROOT / "data" / "wordplay_metadata.csv",
    keep_default_na=False, na_values=[""],
)

# The 9 features used in the regression. selection_firsts_rev and
# selection_lasts_rev are intentionally excluded (out of scope).
WORDPLAY_FEATURES = [
    "double_def",
    "anagram_single_word",
    "anagram_consec_words",
    "hidden_fwd",
    "hidden_rev",
    "selection_firsts",
    "selection_lasts",
    "selection_alt",
    "selection_alt_rev",
]

n_before = len(f_clue_features_df)
f_clue_features_df = f_clue_features_df.merge(
    wordplay[["clue_id"] + WORDPLAY_FEATURES],
    on="clue_id",
    how="left",
)
assert len(f_clue_features_df) == n_before, \
    "Left join changed row count — wordplay_metadata may have duplicate clue_ids."
n_unmatched = f_clue_features_df[WORDPLAY_FEATURES[0]].isna().sum()
assert n_unmatched == 0, f"{n_unmatched} f_clue rows lack wordplay metadata."

# Cast to int so the OLS design matrix sees them as 0/1 numerics rather
# than booleans — pandas/statsmodels both accept either, but int8 keeps
# the dtype consistent with t_capitalized.
for col in WORDPLAY_FEATURES:
    f_clue_features_df[col] = f_clue_features_df[col].astype(int)

# True-count summary so the reader can see frequency of each device.
n_total = len(f_clue_features_df)
print(f"f_clue_features_df: {n_total:,} rows after wordplay join")
print("\nWordplay feature True-counts:")
for col in WORDPLAY_FEATURES:
    n_true = int(f_clue_features_df[col].sum())
    print(f"  {col:25s}  {n_true:>7,}  ({n_true / n_total:.2%})")


### §4b — Binned mean plots

Binned mean plots reveal nonlinear relationships that a single
correlation coefficient would miss — for example, norm might rise with
phrase length up to a point and then plateau. For each of the five
continuous features we divide its range into ~20 equal-width bins,
compute the mean L2 norm and 95% CI per bin, and plot bin midpoints
against the binned means with error bars. `t_capitalized` is binary, so
it gets a simple two-bar comparison.


In [ ]:
# === §4b binned mean plots: 2x3 grid (5 continuous + 1 binary)
def binned_mean_ci(x, y, n_bins=20):
    """Return (centers, means, ci_low, ci_high, counts) for equal-width bins of x.

    95% CI computed as ±1.96 * SE = ±1.96 * std/sqrt(n) per bin. Bins with
    fewer than 2 observations get NaN CIs (suppressed in the plot).
    """
    edges = np.linspace(np.min(x), np.max(x), n_bins + 1)
    # Right edge inclusive on the last bin so the max value isn't dropped.
    bin_idx = np.clip(np.digitize(x, edges) - 1, 0, n_bins - 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    means = np.full(n_bins, np.nan)
    ci_lo = np.full(n_bins, np.nan)
    ci_hi = np.full(n_bins, np.nan)
    counts = np.zeros(n_bins, dtype=int)
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        counts[b] = n
        if n >= 1:
            means[b] = float(y[mask].mean())
        if n >= 2:
            se = float(y[mask].std(ddof=1) / np.sqrt(n))
            ci_lo[b] = means[b] - 1.96 * se
            ci_hi[b] = means[b] + 1.96 * se
    return centers, means, ci_lo, ci_hi, counts


fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fclue_color = MODEL_COLORS["g_stock"]

# Layout (row, col): one panel per feature.
panels = [
    ("t_tokens",      "target subword tokens"),
    ("p_tokens",      "phrase subword tokens"),
    ("t_position",    "target position (0=start, 1=end)"),
    ("t_p_ratio",     "target / phrase token ratio"),
    ("p_punctuation", "punctuation tokens in phrase"),
    ("t_capitalized", "target capitalized (0/1)"),
]

# Stash the t_capitalized bar means here so we can rescale that panel's
# y-axis after all panels are drawn — without an explicit ylim, the bar
# chart defaults to starting at 0, which flattens the small mode difference
# we want to make visible.
t_cap_means = None

for ax_idx, (feat, label) in enumerate(panels):
    ax = axes[ax_idx // 3, ax_idx % 3]
    x = f_clue_features_df[feat].values.astype(float)
    y = f_clue_features_df["norm"].values

    if feat == "t_capitalized":
        # Binary: simple two-bar comparison with 95% CIs from per-class SE.
        cats = [0, 1]
        means_b = []
        cis_b = []
        for c in cats:
            mask = x == c
            n = int(mask.sum())
            mu = float(y[mask].mean())
            se = float(y[mask].std(ddof=1) / np.sqrt(n)) if n >= 2 else 0.0
            means_b.append(mu)
            cis_b.append(1.96 * se)
        ax.bar(cats, means_b, yerr=cis_b, color=fclue_color, alpha=0.6,
               edgecolor=fclue_color, capsize=4, width=0.6)
        ax.set_xticks(cats)
        ax.set_xticklabels(["0 (lower)", "1 (capitalized)"])
        # Save bar means so we can compute a tight y-range below — see the
        # rescale block after the loop.
        t_cap_means = (means_b, cis_b)
    else:
        centers, means, ci_lo, ci_hi, counts = binned_mean_ci(x, y, n_bins=20)
        keep = ~np.isnan(means)
        ax.errorbar(centers[keep], means[keep],
                    yerr=[means[keep] - ci_lo[keep], ci_hi[keep] - means[keep]],
                    fmt="o", color=fclue_color, ecolor=fclue_color,
                    capsize=2, markersize=5, alpha=0.85, linewidth=1)

    ax.set_xlabel(label)
    ax.set_ylabel("mean L2 norm")
    ax.set_title(feat)
    ax.grid(alpha=0.3)

# Rescale the t_capitalized panel's y-axis. Bar charts default to ymin=0,
# which compresses any difference between the two bars (~30 vs ~30) into
# imperceptibility. Use a tight range around the bar tops + their CIs so
# the visual scale matches the binned-mean panels (which span ~1-2 norm
# units each), making the mode difference visible.
if t_cap_means is not None:
    means_b, cis_b = t_cap_means
    lo = min(m - ci for m, ci in zip(means_b, cis_b))
    hi = max(m + ci for m, ci in zip(means_b, cis_b))
    rng = hi - lo if hi > lo else 1.0
    ax_tcap = axes[1, 2]
    ax_tcap.set_ylim(lo - 0.5 * rng, hi + 0.5 * rng)

fig.suptitle("f_clue L2 norm vs surface features (binned means with 95% CI)",
             fontsize=13)
fig.tight_layout()

binned_fig_path = FIG_DIR / "norm_bimodality_surface_features_binned.png"
fig.savefig(binned_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {binned_fig_path}")


These 9 panels show mean L2 norm for True vs False for each wordplay
feature. The y-axes are rescaled to a tight range so small differences
between True and False bars are visible — same approach as the
`t_capitalized` panel above.


In [ ]:
# === §4b wordplay binned-mean plots: 3x3 grid of True-vs-False bar charts
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fclue_color = MODEL_COLORS["g_stock"]

for ax_idx, feat in enumerate(WORDPLAY_FEATURES):
    ax = axes[ax_idx // 3, ax_idx % 3]
    x = f_clue_features_df[feat].values.astype(float)
    y = f_clue_features_df["norm"].values

    # Two-bar comparison: mean L2 norm for False (0) vs True (1) with 95% CIs.
    cats = [0, 1]
    means_b = []
    cis_b = []
    for c in cats:
        mask = x == c
        n = int(mask.sum())
        mu = float(y[mask].mean()) if n >= 1 else float("nan")
        se = float(y[mask].std(ddof=1) / np.sqrt(n)) if n >= 2 else 0.0
        means_b.append(mu)
        cis_b.append(1.96 * se)

    ax.bar(cats, means_b, yerr=cis_b, color=fclue_color, alpha=0.6,
           edgecolor=fclue_color, capsize=4, width=0.6)
    ax.set_xticks(cats)
    ax.set_xticklabels(["0 (False)", "1 (True)"])
    ax.set_xlabel(feat)
    ax.set_ylabel("mean L2 norm")
    ax.set_title(feat)
    ax.grid(alpha=0.3)

    # Tight y-range around bar tops + their CIs so small True-vs-False
    # differences are visible (bar charts default to ymin=0, which
    # flattens any sub-1 norm-unit difference into imperceptibility).
    lo = min(m - ci for m, ci in zip(means_b, cis_b))
    hi = max(m + ci for m, ci in zip(means_b, cis_b))
    rng = hi - lo if hi > lo else 1.0
    ax.set_ylim(lo - 0.5 * rng, hi + 0.5 * rng)

fig.suptitle("f_clue L2 norm by wordplay type (binned means with 95% CI)",
             fontsize=13)
fig.tight_layout()

wordplay_fig_path = FIG_DIR / "norm_bimodality_wordplay_features_binned.png"
fig.savefig(wordplay_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {wordplay_fig_path}")


### §4c — Multiple regression

We fit OLS multiple regression predicting L2 norm from all 15 features
simultaneously (6 surface + 9 wordplay). To make the incremental
contribution of the wordplay flags visible, we also fit a parallel OLS
on the original 6 surface features alone and report both R² values
side-by-side. Standardizing all features to zero mean and unit variance
before fitting makes the resulting βs directly comparable in magnitude
(1 standard deviation of feature change → β standard deviations of norm
change).

With ~239K rows even tiny effects will be statistically significant, so
we focus interpretation on effect *sizes* (β magnitude and CI width
relative to it), not p-values.


In [ ]:
# === §4c OLS multiple regression: standardized βs + 95% CIs (15 features)
import statsmodels.api as sm

reg_features = ["t_tokens", "p_tokens", "t_position", "t_p_ratio",
                "t_capitalized", "p_punctuation"]
# Full feature set: 6 surface features + 9 wordplay features = 15.
reg_features_15 = reg_features + WORDPLAY_FEATURES

y = f_clue_features_df["norm"].astype(float).values


def fit_standardized_ols(features):
    """Fit OLS on standardized features and return (model, raw_model).

    Standardization makes coefficient magnitudes directly comparable: β
    is the change in norm (in norm SD-units? no — in raw norm units) per
    1-SD change in the feature. The raw-coefficient parallel fit is for
    reporting unstandardized coefficients in the results file.
    """
    X_raw_local = f_clue_features_df[features].astype(float).values
    feature_means = X_raw_local.mean(axis=0)
    feature_stds = X_raw_local.std(axis=0, ddof=0)
    # Guard: if a feature has zero variance, leave it as zeros (its
    # coefficient would be undefined). In practice none of these will be flat.
    safe_stds = np.where(feature_stds == 0, 1.0, feature_stds)
    X_std_local = (X_raw_local - feature_means) / safe_stds
    model = sm.OLS(y, sm.add_constant(X_std_local)).fit()
    raw_model = sm.OLS(y, sm.add_constant(X_raw_local)).fit()
    return model, raw_model


ols_t0 = time.time()
# Original 6-feature fit, kept under its own name for the R² comparison.
ols_model_6, _ = fit_standardized_ols(reg_features)
# New 15-feature fit — this becomes the canonical `ols_model` used by
# §4d's coefficient plot, §6a's printed summary, and §6b's results file.
ols_model, ols_raw = fit_standardized_ols(reg_features_15)
print(f"OLS fits in {time.time() - ols_t0:.2f}s")

print(f"\nOriginal 6-feature R²:        {ols_model_6.rsquared:.4f}")
print(f"New 15-feature R²:            {ols_model.rsquared:.4f}")
print(f"Incremental R² from wordplay: {ols_model.rsquared - ols_model_6.rsquared:.4f}")
print(f"Adjusted R² (15 features):    {ols_model.rsquared_adj:.4f}")
print(f"Unexplained variance (15):    {1 - ols_model.rsquared:.4f}")

# Pull standardized βs and their 95% CIs from the 15-feature model
# (statsmodels returns them in the same order as the design matrix; index 0
# is the intercept).
betas_std = ols_model.params[1:]
ci_low = ols_model.conf_int(alpha=0.05)[1:, 0]
ci_high = ols_model.conf_int(alpha=0.05)[1:, 1]
betas_raw = ols_raw.params[1:]

ols_table = pd.DataFrame({
    "Feature": reg_features_15,
    "β (standardized)": betas_std,
    "95% CI low":  ci_low,
    "95% CI high": ci_high,
    "Raw coefficient": betas_raw,
})
# Sort by |β| so the strongest predictors are at the top.
ols_table["abs_beta"] = ols_table["β (standardized)"].abs()
ols_table = ols_table.sort_values("abs_beta", ascending=False).drop(columns="abs_beta")

print("\nStandardized regression coefficients (15 features, sorted by |β|):")
print(ols_table.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))


### §4d — Coefficient plot

Dot-and-whisker plot of the standardized βs with 95% CIs. Features are
sorted by |β| so the strongest predictors sit at the top; the vertical
dashed line at 0 is the no-effect reference. CIs at 239K rows are
narrow, so the visual emphasis falls on relative magnitude rather than
significance.


In [ ]:
# === §4d standardized coefficient dot-and-whisker plot (15 features)
fig, ax = plt.subplots(figsize=(7, 7))

# Plot from bottom to top so the largest |β| sits at the top of the y-axis.
plot_table = ols_table.iloc[::-1].reset_index(drop=True)
y_pos = np.arange(len(plot_table))
err_low  = plot_table["β (standardized)"] - plot_table["95% CI low"]
err_high = plot_table["95% CI high"] - plot_table["β (standardized)"]

ax.errorbar(plot_table["β (standardized)"], y_pos,
            xerr=[err_low, err_high],
            fmt="o", color=MODEL_COLORS["g_stock"],
            ecolor=MODEL_COLORS["g_stock"], capsize=4,
            markersize=8, linewidth=1.5)
ax.axvline(0, color="0.4", linestyle="--", linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_table["Feature"])
ax.set_xlabel("standardized β (norm SDs per feature SD)")
# Title's R² annotation reads from the 15-feature ols_model.
ax.set_title(f"Surface + wordplay coefficients on f_clue L2 norm  "
             f"(R² = {ols_model.rsquared:.3f})")
ax.grid(alpha=0.3, axis="x")

fig.tight_layout()
coef_fig_path = FIG_DIR / "norm_bimodality_surface_features_coefficients.png"
fig.savefig(coef_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {coef_fig_path}")


## §5 — Directional structure in f_clue embeddings

The bimodal L2 norm distribution in f_clue tells us about *magnitude* — that
clue-contextualized embeddings cluster into two preferred lengths — but the
model could *also* encode clue-level properties along directions, not just
along the radial axis. This section asks whether the f_clue embeddings show
clustering or directional structure beyond magnitude. We unit-normalize every
f_clue embedding (so cosine similarity becomes the dot product) and then
sample 50,000 random row pairs and compute their pairwise cosine similarities.
A clearly multi-modal cosine distribution would suggest directional clusters;
a unimodal, roughly Gaussian distribution would mean there is no apparent
directional structure beyond what magnitude already shows.


In [ ]:
# === §5 directional structure: 50K random pairwise cosines on unit-normalized f_clue
# Reload the full f_clue embedding array — only the L2 norm vector was kept
# in memory in §1a (the 1024-dim array was deleted to free ~980 MB). For the
# directional analysis we need the full embeddings back, but only briefly:
# we unit-normalize, sample pairs, compute dot products, and free the array.
dir_t0 = time.time()
fclue_path = EMBED_DIR / "g_stock" / "f_clue.npy"
fclue_emb = np.load(fclue_path)
assert fclue_emb.shape == (239_406, 1024), (
    f"Expected (239406, 1024); got {fclue_emb.shape}"
)

# Unit-normalize every row so cosine_sim(a, b) = a · b. Without this step,
# the dot product would conflate direction with magnitude — and we already
# know magnitudes are bimodal, so the cosine signal would be polluted.
row_norms = np.linalg.norm(fclue_emb, axis=1, keepdims=True)
fclue_unit = fclue_emb / row_norms
del fclue_emb  # free 1024-dim array now that the unit-norm copy exists
print(f"Unit-normalized {fclue_unit.shape[0]:,} f_clue embeddings  "
      f"({time.time() - dir_t0:.1f}s)")

# Sample 50,000 random pairs (i, j) with i != j. Two independent integer
# draws + an explicit i != j filter is faster and simpler than constructing
# unique-pair combinations, and at 50K out of ~239K^2 possible pairs there
# are no meaningful collisions to worry about.
rng_dir = np.random.default_rng(42)
N_PAIRS = 50_000
n_rows = fclue_unit.shape[0]
i_idx = rng_dir.integers(0, n_rows, size=N_PAIRS)
j_idx = rng_dir.integers(0, n_rows, size=N_PAIRS)
# Resample any (i, j) where i == j until disjoint — typically affects only
# a handful of pairs.
collision = i_idx == j_idx
n_resampled = int(collision.sum())
while collision.any():
    j_idx[collision] = rng_dir.integers(0, n_rows, size=int(collision.sum()))
    collision = i_idx == j_idx
print(f"Sampled {N_PAIRS:,} random pairs  "
      f"(resampled {n_resampled} initial i==j collisions)")

# Cosine similarity = dot product between unit-normalized rows.
pair_cos = np.einsum("ij,ij->i", fclue_unit[i_idx], fclue_unit[j_idx])
print(f"Computed {N_PAIRS:,} pairwise cosines  "
      f"(total §5 wall-clock so far: {time.time() - dir_t0:.1f}s)")

# Free the unit-normalized array now that we only need the 50K cosines.
del fclue_unit

# Summary statistics — printed and reused below in the figure annotation.
pair_cos_mean   = float(np.mean(pair_cos))
pair_cos_median = float(np.median(pair_cos))
pair_cos_std    = float(np.std(pair_cos))
print(f"\nPairwise cosine summary (N = {N_PAIRS:,}):")
print(f"  Mean:   {pair_cos_mean:.4f}")
print(f"  Median: {pair_cos_median:.4f}")
print(f"  Std:    {pair_cos_std:.4f}")

# Visualize the distribution as a single-panel KDE — same hatched/edge style
# as §2 so the two figures read as a pair (one for magnitude, one for
# direction). The cosine range is bounded in [-1, 1]; we let the KDE pick
# its own x-range and pad slightly.
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(8, 5))
fill_color = MODEL_COLORS["g_stock"]
kde_pair = gaussian_kde(pair_cos)
x_lo_pair = float(pair_cos.min()) - 0.02 * (float(pair_cos.max()) - float(pair_cos.min()))
x_hi_pair = float(pair_cos.max()) + 0.02 * (float(pair_cos.max()) - float(pair_cos.min()))
x_grid = np.linspace(x_lo_pair, x_hi_pair, 500)
y_kde = kde_pair(x_grid)
ax.fill_between(x_grid, y_kde, alpha=0.45, color=fill_color,
                hatch="//", edgecolor=fill_color)

ax.set_xlabel("pairwise cosine similarity")
ax.set_ylabel("density")
ax.set_title(f"f_clue pairwise cosine similarity (N = {N_PAIRS:,} random pairs)")
ax.set_xlim(x_lo_pair, x_hi_pair)
ax.grid(alpha=0.3)

# Annotation: mean / median / std + N. Same white-bg box style as §2.
ax.annotate(
    f"N = {N_PAIRS:,}\n"
    f"mean   = {pair_cos_mean:.3f}\n"
    f"median = {pair_cos_median:.3f}\n"
    f"std    = {pair_cos_std:.3f}",
    xy=(0.04, 0.96), xycoords="axes fraction",
    ha="left", va="top", fontsize=10, family="monospace",
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.7", alpha=0.9),
)

fig.tight_layout()
pair_cos_fig_path = FIG_DIR / "norm_bimodality_fclue_pairwise_cosine.png"
fig.savefig(pair_cos_fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {pair_cos_fig_path}")
print(f"§5 wall-clock: {time.time() - dir_t0:.1f}s")


The pairwise cosine distribution is plotted above. Read alongside the §2
norm distributions: if it is unimodal and roughly Gaussian, the f_clue
embeddings show no apparent directional clustering beyond what their
bimodal magnitudes already encode — clue-level structure is captured along
the radial axis, not in distinct directional clusters. A clearly multi-modal
cosine distribution would instead suggest directional clusters that the L2
norm view does not reveal.


### §5a — Centroid direction comparison across the bimodal halves

The pairwise cosine view tells us f_clue embeddings have no overall directional
clustering, but it averages over the population — the bimodal *norm* groups
could still face systematically different directions even while the global
cosine distribution looks unimodal. To check this, we split the embeddings
into a lower and upper half at the median L2 norm and compute the cosine
between the two halves' mean unit-direction vectors. A value near 1 means
the two norm groups face the same direction (norm and direction are
disentangled); a meaningfully lower value means the norm dimension and the
directional dimension are at least partly intertwined.


In [ ]:
# === §5a centroid direction comparison: cosine between mean unit vectors of the two norm halves
# Reload the full f_clue embedding array. Cell 45 deleted both the raw and the
# unit-normalized arrays after computing pairwise cosines, so neither is in
# scope here. Reusing norms["f_clue (full)"] (kept from §1a) for the split.
cent_t0 = time.time()
fclue_emb = np.load(EMBED_DIR / "g_stock" / "f_clue.npy")
assert fclue_emb.shape == (239_406, 1024), (
    f"Expected (239406, 1024); got {fclue_emb.shape}"
)

# Split point is the median of the f_clue full norm vector. Lower half =
# norms strictly below the median; upper half = norms at or above. With an
# even N (239,406) the >= split puts the median row itself into the upper
# half, leaving the two halves equal in size by construction.
fclue_norms_full = norms["f_clue (full)"]
median_norm = float(np.median(fclue_norms_full))
lower_mask = fclue_norms_full <  median_norm
upper_mask = fclue_norms_full >= median_norm
n_lower = int(lower_mask.sum())
n_upper = int(upper_mask.sum())
print(f"Median norm split point: {median_norm:.4f}")
print(f"  Lower half (norm <  median): N = {n_lower:,}")
print(f"  Upper half (norm >= median): N = {n_upper:,}")

# Unit-normalize every row so the centroid below is the mean *direction*,
# not pulled toward whichever half has larger raw magnitudes. This is the
# same step cell 45 performed; we redo it here because the prior unit-norm
# array was freed at the end of that cell.
row_norms = np.linalg.norm(fclue_emb, axis=1, keepdims=True)
fclue_unit = fclue_emb / row_norms
del fclue_emb  # free 1024-dim raw array; only the unit-norm copy is needed below

# Mean unit vector per half. The mean of unit vectors is generally NOT itself
# a unit vector — its L2 norm reflects directional concentration: close to 1
# means the half is tightly clustered around its mean direction, close to 0
# means the half is directionally diffuse. We capture both pre-normalization
# norms below before normalizing the centroids for the cosine comparison.
centroid_lower = fclue_unit[lower_mask].mean(axis=0)
centroid_upper = fclue_unit[upper_mask].mean(axis=0)
del fclue_unit  # the two centroids are all we need going forward

centroid_lower_norm = float(np.linalg.norm(centroid_lower))
centroid_upper_norm = float(np.linalg.norm(centroid_upper))

# Normalize each centroid to unit length, then take their dot product. This
# is the cosine between the two centroid *directions*, ignoring how
# concentrated each half is around its mean.
centroid_lower_unit = centroid_lower / centroid_lower_norm
centroid_upper_unit = centroid_upper / centroid_upper_norm
centroid_cosine = float(np.dot(centroid_lower_unit, centroid_upper_unit))

print(f"\nPre-normalization centroid L2 norms (directional concentration):")
print(f"  Lower-half centroid norm: {centroid_lower_norm:.4f}")
print(f"  Upper-half centroid norm: {centroid_upper_norm:.4f}")
print(f"  (close to 1 = tightly clustered direction; "
      f"close to 0 = directionally diffuse)")
print(f"\nCentroid-direction cosine similarity: {centroid_cosine:.4f}")
print(f"  (1.00 = the two halves face the same direction; "
      f"lower values mean norm and direction are entangled)")
print(f"\n§5a wall-clock: {time.time() - cent_t0:.1f}s")


## §6 — Summary and results file

### §6a — Findings summary

Populated by the numerical results above. The findings list summarizes
which populations are bimodal under the two-criterion test, what the
ICC and cross-format correlation reveal about word-level vs contextual
norm structure, and how much of f_clue norm variance the surface
features explain.


In [ ]:
# === §6a print human-readable summary
n_total = len(norms)
n_bimodal = len(bimodal_labels)

# Group bimodal labels by phrase type to see whether bimodality is a
# property of one phrase format or shared across all of them.
phrase_types_bimodal = set()
for lbl in bimodal_labels:
    for ptype in ("f_clue", "f_wndef", "f_wnex"):
        if ptype in lbl:
            phrase_types_bimodal.add(ptype)

# How many populations would have been flagged by ΔBIC alone vs. the
# combined test — useful for showing why Ashman's D matters at large N.
n_passing_dbic = int((delta_bic_df["ΔBIC"] > DELTA_BIC_THRESHOLD).sum())
n_passing_d    = int((delta_bic_df["Ashman's D"] > ASHMANS_D_THRESHOLD).sum())

most_bimodal = delta_bic_df.iloc[0]
least_bimodal = delta_bic_df.iloc[-1]

# Pull D values into locals so the f-strings below don't have nested
# same-style quotes (incompatible with Python <3.12).
most_d  = most_bimodal["Ashman's D"]
least_d = least_bimodal["Ashman's D"]

print(f"Bimodal populations (ΔBIC > {DELTA_BIC_THRESHOLD:g} AND "
      f"D > {ASHMANS_D_THRESHOLD:g}): {n_bimodal} of {n_total}")
print(f"  Passing ΔBIC criterion alone:        {n_passing_dbic} of {n_total}")
print(f"  Passing Ashman's D criterion alone:  {n_passing_d} of {n_total}")
print(f"Phrase types showing bimodality: "
      f"{sorted(phrase_types_bimodal) if phrase_types_bimodal else 'none'}")
print(f"Highest ΔBIC:    {most_bimodal['Population']:25s}  "
      f"ΔBIC = {most_bimodal['ΔBIC']:.1f}, D = {most_d:.2f}")
print(f"Lowest ΔBIC:     {least_bimodal['Population']:25s}  "
      f"ΔBIC = {least_bimodal['ΔBIC']:.1f}, D = {least_d:.2f}")

# Word-subset control summary: status of each wnex-aligned panel.
print("\nWord-subset controls (wnex-aligned populations):")
for ctrl in ["f_clue, def ∈ wnex", "f_wndef, wnex words", "f_wnex (full)"]:
    row = fit_by_label[ctrl]
    bimodal_str = "bimodal" if ctrl in bimodal_labels else "unimodal"
    ctrl_d = row["Ashman's D"]
    print(f"  {ctrl:25s}  ΔBIC = {row['ΔBIC']:>8.1f}  "
          f"D = {ctrl_d:.2f}  ({bimodal_str})")

# §3 — word-vs-context findings.
print("\n§3 — Is L2 norm a property of the word?")
print(f"  ICC(1,1):                              {icc_value:.4f}  ({interp})")
print(f"  Words with >=2 f_clue appearances:     {n_words:,}")
print(f"  Clue rows covered:                     {n_rows:,}")
print(f"  Median within-word std:                {median_within_std:.4f}")
print(f"  Overall population std:                {overall_std:.4f}")
print(f"  Cross-format Spearman ρ (mean f_clue vs f_wndef): "
      f"{rho_fclue_wndef:.3f}  (N = {len(cross_df):,} words)")
print(f"  Cross-format Spearman ρ (mean f_clue vs f_wnex):  "
      f"{rho_fclue_wnex:.3f}")
print(f"  Cross-format Spearman ρ (f_wndef vs f_wnex):      "
      f"{rho_wndef_wnex:.3f}")

# §4 — surface-feature + wordplay regression.
print("\n§4 — Surface + wordplay features predicting f_clue L2 norm")
print(f"  Original 6-feature R²:   {ols_model_6.rsquared:.4f}")
print(f"  15-feature R²:           {ols_model.rsquared:.4f}")
print(f"  Wordplay incremental:    {ols_model.rsquared - ols_model_6.rsquared:.4f}")
top3 = ols_table.head(3)
print("  Top 3 standardized predictors (by |β|, 15-feature model):")
for _, r in top3.iterrows():
    print(f"    {r['Feature']:22s}  β = {r['β (standardized)']:+.4f}  "
          f"95% CI [{r['95% CI low']:+.4f}, {r['95% CI high']:+.4f}]")


### §6b — Write results file

`outputs/cale_norm_bimodality-results.md` consolidates the §1 norm
summary (with min/max), the full ΔBIC + Ashman's D table, the focused
two-criterion bimodality view, the full-vs-val equivalence comparison,
the GMM parameter table for the bimodal populations, the §3 ICC and
cross-format statistics, and the §4 surface-feature regression
coefficients. This is the file the Architect reviews when interpreting
these results.


In [ ]:
# === §6b build outputs/cale_norm_bimodality-results.md
RESULTS_PATH = OUTPUT_DIR / "results" / "cale_norm_bimodality-results.md"

lines = []
lines.append("# CALE L2 Norm Bimodality Survey (g_stock) — Results")
lines.append("")
lines.append(f"Generated: {date.today().isoformat()}")
lines.append("")
lines.append("## Versions")
lines.append("")
for k, v in VERSIONS.items():
    lines.append(f"- **{k}:** {v}")
lines.append("")

# §1 norm summary (now includes Min and Max columns).
lines.append("## §1 — Norm summary (all g_stock populations)")
lines.append("")
norm_md = norm_summary_df.to_markdown(
    index=False,
    floatfmt=("", ",.0f", ".4f", ".4f", ".4f", ".4f"),
)
lines.append(norm_md)
lines.append("")

# §2 full ΔBIC + Ashman's D table — separate float formats for BIC vs
# Ashman's D vs GMM params, so build the markdown by hand rather than
# relying on a single floatfmt.
lines.append("## §2 — ΔBIC and Ashman's D (all populations)")
lines.append("")
hdr = ["Population", "N", "BIC(1)", "BIC(2)", "ΔBIC", "Ashman's D",
       "μ1", "σ1", "π1", "μ2", "σ2", "π2"]
lines.append("| " + " | ".join(hdr) + " |")
lines.append("|" + "|".join(["---"] * len(hdr)) + "|")
for _, r in delta_bic_df.iterrows():
    # Pull "Ashman's D" out of the row first so the f-string below doesn't
    # need nested same-style quotes (Python <3.12 limitation).
    d_val = r["Ashman's D"]
    row = [
        str(r["Population"]),
        f"{int(r['N']):,}",
        f"{r['BIC(1)']:.1f}",
        f"{r['BIC(2)']:.1f}",
        f"{r['ΔBIC']:.1f}",
        f"{d_val:.2f}",
        f"{r['μ1']:.4f}",
        f"{r['σ1']:.4f}",
        f"{r['π1']:.4f}",
        f"{r['μ2']:.4f}",
        f"{r['σ2']:.4f}",
        f"{r['π2']:.4f}",
    ]
    lines.append("| " + " | ".join(row) + " |")
lines.append("")

# §2 focused view — two-criterion bimodality test
lines.append("## §2 — Bimodal populations (two-criterion test)")
lines.append("")
focused_md = bimodal_view_df.to_markdown(
    index=False,
    floatfmt=("", ",.0f", ".1f", ".2f", ""),
)
lines.append(focused_md)
lines.append("")
lines.append(f"Bimodal criteria: ΔBIC > {DELTA_BIC_THRESHOLD:g} "
             '(Kass & Raftery 1995, "very strong" evidence) AND '
             f"Ashman's D > {ASHMANS_D_THRESHOLD:g} "
             '(Ashman, Bird & Zepf 1994, "cleanly separated" components).')
lines.append("")
lines.append("ΔBIC scales with sample size, so at large N it flags any "
             "minor deviation from normality. Ashman's D is sample-size-"
             "independent and provides the effect-size filter that "
             "distinguishes meaningful bimodality from sample-size artefact.")
lines.append("")

# §2 full vs val equivalence check
lines.append("## §2 — Full vs val equivalence")
lines.append("")
fullval_md = fullval_df.to_markdown(
    index=False,
    floatfmt=("", "", ",.0f", ".2f", ".4f", ".4f", ".4f", ".4f"),
)
lines.append(fullval_md)
lines.append("")
lines.append("Full and val agree on Ashman's D and on all four GMM "
             "component parameters. The annotated display figure (§2d) "
             "therefore shows only the full-dataset version of each phrase "
             "type.")
lines.append("")

# §2 GMM parameters for bimodal populations only
lines.append("## §2 — GMM parameters (bimodal populations)")
lines.append("")
if bimodal_labels:
    gmm_only = delta_bic_df[
        delta_bic_df["Population"].isin(bimodal_labels)
    ][["Population", "N", "ΔBIC", "Ashman's D",
       "μ1", "σ1", "π1", "μ2", "σ2", "π2"]]
    gmm_md = gmm_only.to_markdown(
        index=False,
        floatfmt=("", ",.0f", ".1f", ".2f",
                  ".4f", ".4f", ".4f", ".4f", ".4f", ".4f"),
    )
    lines.append(gmm_md)
else:
    lines.append("_No populations passed both bimodality criteria; GMM "
                 "parameters omitted._")
lines.append("")

# §3 — Is L2 norm a property of the word?
lines.append("## §3 — Is L2 norm a property of the word?")
lines.append("")
lines.append(f"- **ICC(1,1):** {icc_value:.4f}  ({interp})")
lines.append(f"- **Words with ≥2 appearances:** {n_words:,}")
lines.append(f"- **Clue rows covered:** {n_rows:,}")
lines.append(f"- **Avg group size (k):** {k_bar:.2f}")
lines.append(f"- **Median within-word std:** {median_within_std:.4f}")
lines.append(f"- **Overall population std:** {overall_std:.4f}")
lines.append(f"- **Cross-format Spearman ρ (mean f_clue vs f_wndef):** "
             f"{rho_fclue_wndef:.3f}")
lines.append(f"- **Cross-format Spearman ρ (mean f_clue vs f_wnex):** "
             f"{rho_fclue_wnex:.3f}")
lines.append(f"- **Cross-format Spearman ρ (f_wndef vs f_wnex):** "
             f"{rho_wndef_wnex:.3f}")
lines.append(f"- **N words in cross-format comparison:** {len(cross_df):,}")
lines.append("")

# §4 — Surface features predicting f_clue L2 norm
lines.append("## §4 — Surface + wordplay features predicting f_clue L2 norm")
lines.append("")
lines.append(f"- **Original 6-feature R²:** {ols_model_6.rsquared:.4f}")
lines.append(f"- **New 15-feature R²:** {ols_model.rsquared:.4f}")
lines.append(f"- **Incremental R² from wordplay:** {ols_model.rsquared - ols_model_6.rsquared:.4f}")
lines.append(f"- **Adjusted R² (15 features):** {ols_model.rsquared_adj:.4f}")
lines.append(f"- **Variance unexplained (1 − R², 15 features):** {1 - ols_model.rsquared:.4f}")
lines.append(f"- **N rows:** {len(f_clue_features_df):,}")
lines.append("")
hdr_ols = ["Feature", "β (standardized)", "95% CI", "Raw coefficient"]
lines.append("| " + " | ".join(hdr_ols) + " |")
lines.append("|" + "|".join(["---"] * len(hdr_ols)) + "|")
for _, r in ols_table.iterrows():
    ci_str = f"[{r['95% CI low']:+.4f}, {r['95% CI high']:+.4f}]"
    lines.append(
        f"| {r['Feature']} | {r['β (standardized)']:+.4f} | {ci_str} | "
        f"{r['Raw coefficient']:+.4f} |"
    )
lines.append("")

# §5 — Directional structure in f_clue embeddings
lines.append("## §5 — Directional structure in f_clue embeddings")
lines.append("")
lines.append(f"- **N pairs sampled:** {N_PAIRS:,}")
lines.append(f"- **Mean cosine similarity:** {pair_cos_mean:.4f}")
lines.append(f"- **Median cosine similarity:** {pair_cos_median:.4f}")
lines.append(f"- **Std:** {pair_cos_std:.4f}")
lines.append("")
lines.append("### §5a — Centroid direction comparison across the bimodal halves")
lines.append("")
lines.append(f"- **Median norm split point:** {median_norm:.4f}")
lines.append(f"- **Lower-half size (norm < median):** {n_lower:,}")
lines.append(f"- **Upper-half size (norm ≥ median):** {n_upper:,}")
lines.append(f"- **Lower-half centroid pre-normalization L2 norm:** "
             f"{centroid_lower_norm:.4f}")
lines.append(f"- **Upper-half centroid pre-normalization L2 norm:** "
             f"{centroid_upper_norm:.4f}")
lines.append(f"- **Centroid-direction cosine similarity:** "
             f"{centroid_cosine:.4f}")
lines.append("")
lines.append("Centroid pre-normalization L2 norm reflects directional "
             "concentration within the half (close to 1 = tightly clustered "
             "around the mean direction; close to 0 = directionally diffuse). "
             "The centroid-direction cosine answers whether the two halves "
             "face the same direction.")
lines.append("")
lines.append("Figure: `outputs/figures/norm_bimodality_fclue_pairwise_cosine.png`")
lines.append("")

# Figures
lines.append("## Figures")
lines.append("")
lines.append("- `outputs/figures/norm_bimodality_survey_raw.png` — raw "
             "3×3 KDE grid, all 7 populations grouped by phrase type "
             "(rows) × variant (columns), shared x-axis")
lines.append("- `outputs/figures/norm_bimodality_survey_gstock.png` — "
             "annotated 3×2 display: full-dataset (left) vs wnex-aligned "
             "subset (right) for each phrase type, with GMM overlays on "
             "bimodal panels and mean lines on unimodal panels")
lines.append("- `outputs/figures/norm_bimodality_icc_strip.png` — "
             "per-word f_clue norm strips for ~25 words spanning the "
             "f_clue mean-norm range, with f_wndef diamond and f_wnex "
             "triangle markers and the f_clue GMM mode reference lines")
lines.append("- `outputs/figures/norm_bimodality_cross_format_scatter.png`"
             " — 1×3 panel showing all three pairwise cross-format "
             "norm correlations (mean f_clue vs f_wndef, mean f_clue "
             "vs f_wnex, f_wndef vs f_wnex) on the wnex vocabulary, "
             "each with OLS trend and Spearman ρ annotation")
lines.append("- `outputs/figures/norm_bimodality_surface_features_binned.png`"
             " — binned-mean ± 95% CI panels, one per surface feature")
lines.append("- `outputs/figures/norm_bimodality_wordplay_features_binned.png`"
             " — 9-panel bar chart of mean L2 norm by True/False for "
             "each wordplay feature, with rescaled y-axes for visibility "
             "of small effects")
lines.append("- `outputs/figures/norm_bimodality_surface_features_coefficients.png`"
             " — standardized OLS coefficient dot-and-whisker plot")
lines.append("- `outputs/figures/norm_bimodality_fclue_pairwise_cosine.png`"
             " — pairwise cosine similarity KDE on 50,000 random pairs of "
             "unit-normalized f_clue embeddings")
lines.append("")

RESULTS_PATH.write_text("\n".join(lines))
print(f"Wrote: {RESULTS_PATH}")
print(f"  {len(lines):,} lines")


In [ ]:
# === Wall-clock runtime
elapsed = time.time() - NOTEBOOK_T0
print(f"Notebook wall-clock runtime: {elapsed:.1f}s ({elapsed/60:.2f} min)")

## §7 — Summary

This notebook surveyed L2 norm distributions across **seven** g_stock
embedding populations to identify which exhibit genuine bimodality under
a formal, two-criterion test (ΔBIC > 10 AND Ashman's D > 2), then
investigated whether that bimodality is a property of the word being
embedded or of the surrounding context (§3) and which surface features
of the f_clue input predict the L2 norm of the output (§4).

**Why two criteria, not one.** ΔBIC measures whether the data
statistically prefer a 2-component fit, but it scales with sample size:
at N in the tens of thousands, even minor deviations from perfect
normality push ΔBIC well past the Kass & Raftery threshold. Ashman's D
is a sample-size-independent effect-size measure of how cleanly the two
component means are separated relative to their combined spread, and
D > 2 corresponds to "cleanly separated" components. Combining the two
criteria filters out sample-size artefacts.

**Coverage of the survey:**
- 5 directly loaded populations: f_clue (full + val), f_wndef
  (full + val), f_wnex (full)
- 2 sliced subset populations: f_clue restricted to wnex-vocabulary
  definitions, f_wndef restricted to wnex words
- f_wnex (val) was deliberately omitted (small subset of full wnex)

**§2 results — see the §2b focused view and §6a printout** for the full
classification. The §2c full-vs-val table confirms that validation
slices match the full distributions on Ashman's D and all four GMM
component parameters. The §2d wnex-aligned column isolates whether
bimodality is driven by *which words* or by *how the phrase is
constructed*.

**§3 — Is L2 norm a property of the word?** ICC(1,1) on f_clue norms
asks how much of the total norm variance is between-word vs
within-word; the three pairwise cross-format Spearman ρ values
(mean f_clue vs f_wndef, mean f_clue vs f_wnex, and f_wndef vs
f_wnex) ask whether word-level rank ordering is preserved across
phrase constructions. The §3a example clues for "european",
"style", and "song" show concretely what lower-mode and upper-mode
f_clue norms look like for the same word. See §6a and the results
file for the actual ICC, ρ values, and the strip plot. A low ICC
and near-zero cross-format ρ together would indicate the f_clue
bimodality is contextual rather than a property of word identity.

**§4 — Surface + wordplay features predicting f_clue L2 norm.** A
multiple regression on 15 features of the f_clue clue — six
tokenizer-level surface features (`t_tokens`, `p_tokens`,
`t_position`, `t_p_ratio`, `t_capitalized`, `p_punctuation`) plus nine
binary wordplay flags (`double_def`, `anagram_single_word`,
`anagram_consec_words`, `hidden_fwd`, `hidden_rev`,
`selection_firsts`, `selection_lasts`, `selection_alt`,
`selection_alt_rev`) — reports both the original 6-feature R² and the
full 15-feature R², so the incremental contribution of wordplay
structure is visible. Standardized βs (sorted by |β|) and their 95%
CIs are reported in the dot-and-whisker plot. 1 − R² is what remains
unexplained and may reflect deeper contextual properties not captured
by surface tokenization or wordplay-device flags.

**§5 — Directional structure in f_clue embeddings.** A pairwise cosine
analysis on 50,000 random pairs of unit-normalized f_clue embeddings probes
whether the model encodes clue properties along directions in addition to
magnitude. The §5 KDE figure shows the resulting cosine distribution and
the printed mean / median / std summarize its shape; comparing this view
against the §2 norm distributions tells us whether clue-level structure
appears as directional clusters or only as magnitude variation. §5a then
splits the embeddings at the median L2 norm and reports the cosine between
the two halves' mean unit-direction vectors — a value near 1 means the
norm groups face the same direction (norm and direction are disentangled),
while a meaningfully lower value means the two are partly intertwined.

**Outputs produced:**
- `outputs/figures/norm_bimodality_survey_raw.png` — raw 3×3 KDE grid
- `outputs/figures/norm_bimodality_survey_gstock.png` — annotated 3×2
  display with GMM overlays
- `outputs/figures/norm_bimodality_icc_strip.png` — per-word strip plot
- `outputs/figures/norm_bimodality_cross_format_scatter.png` — 1×3
  panel of all three pairwise cross-format norm scatters
  (mean f_clue vs f_wndef, mean f_clue vs f_wnex, f_wndef vs f_wnex)
- `outputs/figures/norm_bimodality_surface_features_binned.png` —
  6-panel binned-mean plots (surface features)
- `outputs/figures/norm_bimodality_wordplay_features_binned.png` —
  9-panel True-vs-False bar charts (wordplay features), rescaled
  y-axes for visibility of small effects
- `outputs/figures/norm_bimodality_surface_features_coefficients.png`
  — OLS coefficient dot-and-whisker plot
- `outputs/figures/norm_bimodality_fclue_pairwise_cosine.png` — KDE of
  50,000 pairwise cosine similarities on unit-normalized f_clue embeddings
  (directional structure check)
- `outputs/cale_norm_bimodality-results.md` — versions, norm summary,
  full ΔBIC + Ashman's D table, focused two-criterion bimodality view,
  full-vs-val equivalence, GMM parameters for bimodal populations, §3
  ICC + cross-format statistics, §4 OLS coefficients, and figure list

**What this enables next:**
- The bimodal populations (passing both criteria) are candidates for
  the deeper characterization work the previous f_clue notebook
  performed (subsetting by definition position, source, etc.).
- The §3 result clarifies whether word-level interventions (e.g.,
  reweighting or filtering specific definition words) can plausibly
  shift the bimodality, or whether the bimodality is fundamentally
  contextual and only context-level interventions will move it.
- The §4 result quantifies how much of the bimodality is "trivially"
  explained by surface features the model would always have access to —
  setting a floor on the variance any deeper representational story
  needs to explain.
